In [121]:
import os
import sys
import anndata as ad
import scipy
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import scipy.io as sio
import scanpy.external as sce
import matplotlib.pyplot as plt
import re
import gseapy as gp
import anndata as ad
import statistics
import tempfile
import sklearn
import cosg
import leidenalg
import celltypist
import muon as mu
from tqdm import tqdm
sc._settings.n_jobs= 24
sc.settings.verbosity = 1
# Adjust Scanpy figure defaults
sc.settings.set_figure_params(dpi=100, fontsize=10, dpi_save=400,
    facecolor = 'white', figsize=(8,8), format='png')
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)

In [122]:
def check_dict_duplicates(dict):
    seen = set()
    dup = any(item in seen or seen.add(item) for lst in dict.values() for item in lst)
    if dup==False:
        return '无重复'
    else:
        return '有重复'

In [123]:
obj_path = '/home/liyanguo/MyImmuCell/05_MyImmuCell_subpopulation/Level2_Refine_R6/'

In [124]:
finnal_path = '/home/liyanguo/MyImmuCell/05_MyImmuCell_subpopulation/Finnal/'

In [125]:
leiden_groups=['L4_leiden_TOTALVI_0.1','L4_leiden_TOTALVI_0.5', 'L4_leiden_TOTALVI_1', 'L4_leiden_TOTALVI_1.5', 'L4_leiden_TOTALVI_0.3','L4_leiden_TOTALVI_0.8']

In [126]:
def get_norm_annotation_data(celltype):
    adata = sc.read_h5ad(f"{obj_path}/{celltype}/{celltype}_count_scRNA.h5ad")
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    adt = sc.read_h5ad(f"{obj_path}/{celltype}/{celltype}_preprocess_scADT.h5ad")
    adata.obsm = adt.obsm
    
    # read Level1, tcr, bcr infor
    indices = pd.read_csv(f'{obj_path}/{celltype}/R6_indices_{celltype}.csv',index_col=0)
    # join leiden groups
    leiden_data = adt.obs.loc[:,leiden_groups]
    adata.obs = adata.obs.join(indices, how='left')
    adt.obs = adt.obs.join(indices, how='left')
    adata.obs = adata.obs.join(leiden_data, how='left')
    return adata,adt,leiden_data

# 1. 定 终 CD4 Treg Cell Refine

In [127]:
celltype="TregCD4"
R_data_path = f"{obj_path}{celltype}"
sc.settings.figdir=f"{obj_path}{celltype}"

In [128]:
adata,adt,leiden_data = get_norm_annotation_data(celltype)

In [ ]:
sc.pl.umap(adata, color=['Celltype_L1_L2','Celltype_L1_L2_Refine','Celltype_L2_L3_Refine','Celltype_L3_L4_Refine'],
           frameon=False,
           legend_fontsize=4, legend_fontoutline=2,
           size=4)

In [129]:
groupby = "L4_leiden_TOTALVI_0.8"

In [189]:
xlsx = pd.ExcelWriter(f"{obj_path}{celltype}/{groupby}_freq_table.xlsx")
for group in ['SampleID', 'DonorID','scDblFinder.class',
              'Immune_All_Low', 'Adult_Human_Blood', 'Adult_Human_Bone_marrow',
              'AIFI_L2', 'AIFI_L3', 'predicted.celltype.l2',
              
              ]:
    freq_table_multi = adata.obs.groupby([groupby, group]).size()
    pd.DataFrame(freq_table_multi).to_excel(xlsx,sheet_name=group)
xlsx.close()

In [ ]:
#数据簇间信息比较
fig, axs = plt.subplots(5, 2, figsize=(18, 12),constrained_layout=True)
plt.subplots_adjust(hspace=1,wspace=1)
sc.pl.violin(adata, keys='nCount_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,0], show=False)
sc.pl.violin(adata, keys='nFeature_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,1], show=False)
sc.pl.violin(adata, keys='log10GenesPerUMI', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,0], show=False)
sc.pl.violin(adata, keys='percent_top50', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,1], show=False)

sc.pl.violin(adata, keys='percent_apop', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,0], show=False)
sc.pl.violin(adata, keys='percent_ribo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,1], show=False)
sc.pl.violin(adata, keys='percent_ieg', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,0], show=False)
sc.pl.violin(adata, keys='percent_oxphos', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,1], show=False)
sc.pl.violin(adata, keys='percent_hemo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,0], show=False)
sc.pl.violin(adata, keys='G2M.Score', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,1], show=False)

fig.tight_layout()
plt.savefig(f'{obj_path}{celltype}/{groupby}_TOTALVI_L3_metadata.png')

In [191]:
sc.tl.dendrogram(adata,groupby=groupby,use_rep='X_TOTALVI')
sc.tl.dendrogram(adt,groupby=groupby,use_rep='X_TOTALVI')

In [ ]:
#cosg差异
cosg.cosg(adata, key_added=f'cosg_{groupby}', groupby=groupby,
          mu=10,n_genes_user=100,remove_lowly_expressed=True,
         )
df_tmp = pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'])
df_tmp.to_csv(f"{obj_path}{celltype}/cosg_{groupby}.csv")
#cosg差异作图
df_tmp=pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'][:8,]).T
df_tmp=df_tmp.reindex(adata.uns['dendrogram_'+groupby]['categories_ordered'])
marker_genes_list={idx: list(row.values) for idx, row in df_tmp.iterrows()}
marker_genes_list = {k: v for k, v in marker_genes_list.items() if not any(isinstance(x, float) for x in v)}
sc.pl.dotplot(adata, marker_genes_list,
             groupby=groupby,
             dendrogram=True,
             swap_axes=False,
             standard_scale='var',
             save=f'cosg_{groupby}',
             cmap='Spectral_r')

In [ ]:
#Th17
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'LTK','PTPN13','PDE4D','CCR6','RORC','NR1D1','CTSH','KIF5C','LGALS3','USP10','CMTM6','TOB1',
                     'TNFSF13B','CISH','AQP3','AUTS2','NSG1','S100A4'],
              standard_scale='var',groupby=groupby)

In [ ]:
#Th1/Th17
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'GZMH','IL18RAP','S1PR5','LYAR','NKG7','CST7','PRF1','TBX21','LINC01871','KLRG1','MYBL1','EOMES',
                     'EFHD2','DUSP2','SAMD3','CTSW','ID2','MATK','HOPX',],
              standard_scale='var',groupby=groupby)

In [ ]:
#Th1
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'CMC1','CST7','FCRL3','CCL4','SLAMF7','EOMES','PDCD1','NKG7','CCR5','KLRK1','F2R','PLEK'],
              standard_scale='var',groupby=groupby)

In [ ]:
#Th22
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'CRIP1','LGALS1','LGALS1','S100A10','S100A4','PI16','LMNA','ANXA5','ANXA2'],
              standard_scale='var',groupby=groupby)

In [ ]:
#Th2
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'PTGDR2','SNED1','NEFL','GATA3','FXYD7','C1orf162','GDPD5','IL4R','CAPG',
                     'LGALS1','TNFSF10','TNFRSF4','PPP1R9B','CSGALNACT1','NIBAN1','ERN1','SORL1','RUNX2'],
              standard_scale='var',groupby=groupby)

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,0],show=False,layer='denoised_protein')
sc.pl.umap(adata, color='GZMK',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,1],show=False)
sc.pl.umap(adata, color='CCR4',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,2],show=False)
sc.pl.umap(adata, color='CXCR5',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,3],show=False)
sc.pl.umap(adata, color='CTLA4',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,0],show=False)
sc.pl.umap(adata, color='KLRB1',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,1],show=False)
sc.pl.umap(adata, color='CXCR3',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,2],show=False)
sc.pl.umap(adata, color='CCR6',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,3],show=False)
plt.show()

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,0],show=False)
sc.pl.umap(adata, color='IKZF2',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,1],show=False)
sc.pl.umap(adata, color='CTLA4',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,2],show=False)
sc.pl.umap(adata, color='FOXP3',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,3],show=False)
sc.pl.umap(adata, color='CTLA4',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,0],show=False)
sc.pl.umap(adata, color='PDCD1',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,1],show=False)
sc.pl.umap(adata, color='TIGIT',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,2],show=False)
sc.pl.umap(adata, color='IL2RA',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,3],show=False)
plt.show()

In [ ]:
sc.pl.umap(adata, color='TRIB2',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data')

In [ ]:
sc.pl.dotplot(adata, ['ISG15','CD160','KLRD1','IL2RA','CTLA4','FOXP3','FCRL3','CCR10','RORC','KLRB1','HLA-DRB1',
                     'HAVCR2','LAG3','IKZF1','IKZF2','IKZF3','TIGIT','ENTPD3','IL7R','TNFRSF18','BMI1','BCL6','CD44'],
              standard_scale='obs',groupby=groupby,dendrogram=True)

In [ ]:
sc.pl.dotplot(adata, ['CD40LG','CCR4','CCR6','CXCR5','KLRB1','GZMK','CCL5','BTBD9','CXCR3',
                      'GATA3','CCR10','LIMS1','RORC','CD27','CCR1','CCR2','PRDM1',"FAS"],
              standard_scale='obs',groupby=groupby)

In [ ]:
sc.pl.violin(adata,["IKZF2",'IKZF3','FOXP3'],groupby='L4_leiden_TOTALVI_1.5',size=0)

In [ ]:
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMB','BTBD9','GATA3',
                      'LIMS1','RORC','CD27','CCR1','CCR2','PRDM1','CD27','CD28','CXCR5','KLRB1',
                      'TCF7','ICOS','CD27','CXCR3','PDCD1','TBX21','IL6R','TIGIT','LAG3','HAVCR2',
                      'CCL5','CTLA4','CD40LG'],
              standard_scale='obs',groupby=groupby)

In [ ]:
adata.obs['cluster_dummy']="NOT"
adata.obs.loc[adata.obs['L4_leiden_TOTALVI_1.5']=="6",'cluster_dummy'] = "YES"
sc.pl.umap(adata, color='cluster_dummy')

In [130]:
adata.obs[groupby].value_counts()

L4_leiden_TOTALVI_0.8
3    86250
6    80170
7    76314
2    39712
1    33061
0    31848
5    14793
4     6833
Name: count, dtype: int64

In [131]:
#Naïve CD4+ T 需要CD45RA+ CCR7=CD197hi CD62Lhi
# effector memory T cells(TEM, CD45RA-/CCR7-)
# TEMRA cells, which are T cells that re-express CD45RA(CD45RA+/CCR7-)

#IRF4- TGFB1+ CXCR4lo
cell_dict = {'Naïve CD4+ Treg':['7'],#CD45RA+
             'Memory CD4+ Treg':['1','2','3','6'
                          ],
             'HLA-DRhi CD4+ Treg':['5',],#
             'KLRB1+ CD4+ Treg':['0',],#
             'Doublet|Lowquality':['4',#CSF3R
                         ],
            }
#Single-cell atlas of healthy human blood unveils agerelated loss of NKG2C+GZMB–CD8+ memory T cells and accumulation of type 2 memory T cells

#naïve (CD45RApos), memory (CD45RAnegCD73neg), TR1‐like (CD73pos), and activated (HLA‐DRposCD39pos). 

In [132]:
check_dict_duplicates(cell_dict)

'无重复'

In [133]:
# Generate new assignments
for i in cell_dict.keys():
    ind = pd.Series(adata.obs[groupby]).isin(cell_dict[i])
    adata.obs.loc[ind,'Celltype_L4_L5_Refine_R3'] = i

In [134]:
(adata.obs['Celltype_L4_L5_Refine_R3'].isna()).value_counts()

Celltype_L4_L5_Refine_R3
False    368981
Name: count, dtype: int64

In [135]:
adata.obs['Celltype_L4_L5_Refine_R3'].value_counts()

Celltype_L4_L5_Refine_R3
Memory CD4+ Treg      239193
Naïve CD4+ Treg        76314
KLRB1+ CD4+ Treg       31848
HLA-DRhi CD4+ Treg     14793
Doublet|Lowquality      6833
Name: count, dtype: int64

In [136]:
adata = adata[adata.obs['Celltype_L4_L5_Refine_R3'] != "Doublet|Lowquality",:]

In [137]:
adata.obs['Celltype_L4_L5_Refine_R3'].value_counts()

Celltype_L4_L5_Refine_R3
Memory CD4+ Treg      239193
Naïve CD4+ Treg        76314
KLRB1+ CD4+ Treg       31848
HLA-DRhi CD4+ Treg     14793
Name: count, dtype: int64

In [138]:
indices = adata.obs.loc[:,['Celltype_L1_L2','Celltype_L1_L2_Refine','Celltype_L2_L3_Refine',
                           'Celltype_L3_L4_Refine','Celltype_L4_L5_Refine',
                           'Celltype_L4_L5_Refine_R2','Celltype_L4_L5_Refine_R3',
                           groupby,'receptor_type','receptor_type_BCR']]
indices['UMAP_1'] = adata.obsm['X_umap'][:, 0].copy()
indices['UMAP_2'] = adata.obsm['X_umap'][:, 1].copy()
indices.rename(columns={groupby: 'leiden_cluster'}, inplace=True) #Save the cluster categorical, check the relationship between clusters and clinical to avoid missing someone.
indices['leiden_cluster'] = celltype + " c" + indices['leiden_cluster'].astype(str)
os.makedirs(f'{finnal_path}/{celltype}', exist_ok=True)
indices.to_csv(f"{finnal_path}/{celltype}/Finnal_indices_{celltype}.csv")
indices.head()

,Celltype_L1_L2,Celltype_L1_L2_Refine,Celltype_L2_L3_Refine,Celltype_L3_L4_Refine,Celltype_L4_L5_Refine,Celltype_L4_L5_Refine_R2,Celltype_L4_L5_Refine_R3,leiden_cluster,receptor_type,receptor_type_BCR,UMAP_1,UMAP_2
D0589_Rep1_ACATTGGC_AACGCTTA_TGGAACAA,CD4+ T,Naïve CD4+ T,Treg,Treg CD4+,Treg memory CD4+ T,Treg memory CD4+ T,KLRB1+ CD4+ Treg,TregCD4 c0,TCR,NaN,6.445717,4.832929
D0864_M_Rep2_GATAGACA_AACCGAGA_GACAGTGC,CD4+ T,Naïve CD4+ T,Treg,Tem CD4+ T(Treg),Treg Naive CD4+ T,Treg Naïve CD4+ T,Memory CD4+ Treg,TregCD4 c6,TCR,NaN,0.630055,5.215746
D0864_M_Rep2_CCTCTATC_AACGCTTA_CTAAGGTC,CD4+ T,Naïve CD4+ T,Treg,Treg CD4+,Treg Naive CD4+ T,Treg Naïve CD4+ T,Memory CD4+ Treg,TregCD4 c6,TCR,NaN,-1.776781,8.478098
D0878_Rep2_AGCCATGC_TATCAGCA_AACTCACC,CD4+ T,Naïve CD4+ T,Treg,Tem CD4+ T(Treg),Treg Naive CD4+ T,Treg memory CD4+ T,Memory CD4+ Treg,TregCD4 c6,TCR,NaN,0.850640,3.411033
D0878_Rep2_AACTCACC_CACTTCGA_GAGCTGAA,CD4+ T,Naïve CD4+ T,Treg,Treg CD4+,Treg Naive CD4+ T,Treg Naïve CD4+ T,Memory CD4+ Treg,TregCD4 c6,NaN,NaN,0.499229,4.027567


# 2. 定 终 CD8 Treg Cell Refine

In [139]:
celltype="TregCD8"
R_data_path = f"{obj_path}{celltype}"
sc.settings.figdir=f"{obj_path}{celltype}"

In [140]:
adata,adt,leiden_data = get_norm_annotation_data(celltype)

In [ ]:
sc.pl.umap(adata, color=['Celltype_L2_L3_Refine','Celltype_L3_L4_Refine','receptor_type','Celltype_L4_L5_Refine'],
           frameon=False,
           legend_fontsize=4, legend_fontoutline=2,
           size=4)

In [141]:
groupby = "L4_leiden_TOTALVI_0.1"

In [142]:
adata.obs[groupby].value_counts()

L4_leiden_TOTALVI_0.1
0    715
2    625
1    496
3    442
4    348
5    201
Name: count, dtype: int64

In [143]:
#Naïve CD4+ T 需要CD45RA+ CCR7=CD197hi CD62Lhi
# effector memory T cells(TEM, CD45RA-/CCR7-)
# TEMRA cells, which are T cells that re-express CD45RA(CD45RA+/CCR7-)

#IRF4- TGFB1+ CXCR4lo 
cell_dict = {
    'CD8+ Treg':['0','1','2','3','4','5',],#
            }
#Single-cell atlas of healthy human blood unveils agerelated loss of NKG2C+GZMB–CD8+ memory T cells and accumulation of type 2 memory T cells

#naïve (CD45RApos), memory (CD45RAnegCD73neg), TR1‐like (CD73pos), and activated (HLA‐DRposCD39pos). 

In [144]:
check_dict_duplicates(cell_dict)

'无重复'

In [145]:
# Generate new assignments
for i in cell_dict.keys():
    ind = pd.Series(adata.obs[groupby]).isin(cell_dict[i])
    adata.obs.loc[ind,'Celltype_L4_L5_Refine_R3'] = i

In [146]:
(adata.obs['Celltype_L4_L5_Refine_R3'].isna()).value_counts()

Celltype_L4_L5_Refine_R3
False    2827
Name: count, dtype: int64

In [147]:
adata.obs['Celltype_L4_L5_Refine_R3'].value_counts()

Celltype_L4_L5_Refine_R3
CD8+ Treg    2827
Name: count, dtype: int64

In [148]:
indices = adata.obs.loc[:,['Celltype_L1_L2','Celltype_L1_L2_Refine','Celltype_L2_L3_Refine',
                           'Celltype_L3_L4_Refine','Celltype_L4_L5_Refine',
                           'Celltype_L4_L5_Refine_R2','Celltype_L4_L5_Refine_R3',
                           groupby,'receptor_type','receptor_type_BCR']]
indices['UMAP_1'] = adata.obsm['X_umap'][:, 0].copy()
indices['UMAP_2'] = adata.obsm['X_umap'][:, 1].copy()
indices.rename(columns={groupby: 'leiden_cluster'}, inplace=True) #Save the cluster categorical, check the relationship between clusters and clinical to avoid missing someone.
indices['leiden_cluster'] = celltype + " c" + indices['leiden_cluster'].astype(str)
os.makedirs(f'{finnal_path}/{celltype}', exist_ok=True)
indices.to_csv(f"{finnal_path}/{celltype}/Finnal_indices_{celltype}.csv")
indices.head()

,Celltype_L1_L2,Celltype_L1_L2_Refine,Celltype_L2_L3_Refine,Celltype_L3_L4_Refine,Celltype_L4_L5_Refine,Celltype_L4_L5_Refine_R2,Celltype_L4_L5_Refine_R3,leiden_cluster,receptor_type,receptor_type_BCR,UMAP_1,UMAP_2
D0428_Rep2_CCTCCTGA_AACTCACC_AAGAGATC,CD4+ T,Naïve CD4+ T,Treg,Treg CD8+,Treg CD8+ T,Treg CD8+ T,CD8+ Treg,TregCD8 c0,TCR,NaN,1.530939,14.118806
D0590_M_Rep1_ATGCCTAA_AAGGTACA_AACAACCA,CD4+ T,Naïve CD4+ T,Treg,Treg CD8+,Treg CD8+ T,Treg CD8+ T,CD8+ Treg,TregCD8 c1,TCR,NaN,-3.876978,2.590798
D0295_Rep2_TTCACGCA_CTCAATGA_ATCCTGTA,CD4+ T,Naïve CD4+ T,Treg,Treg CD8+,Treg CD8+ T,Treg CD8+ T,CD8+ Treg,TregCD8 c1,TCR,NaN,-1.438534,2.040453
D0690_Rep2_CGACTGGA_CCGTGAGA_ACGTATCA,CD4+ T,Naïve CD4+ T,Treg,Treg CD8+,Treg CD8+ T,Treg CD8+ T,CD8+ Treg,TregCD8 c0,NaN,NaN,2.181204,14.606707
D0058_Rep1_TGAAGAGA_ACACGACC_ACATTGGC,CD4+ T,Naïve CD4+ T,Treg,Treg CD8+,Treg CD8+ T,Treg CD8+ T,CD8+ Treg,TregCD8 c0,NaN,NaN,0.787973,12.277690


# 3. 定 Th1 17 2 22 T Cell Refine

In [62]:
celltype="Th1_17_2_22"
R_data_path = f"{obj_path}{celltype}"
sc.settings.figdir=f"{obj_path}{celltype}"

In [ ]:
adata,adt,leiden_data = get_norm_annotation_data(celltype)

In [ ]:
sc.pl.umap(adata, color=['Celltype_L2_L3_Refine','Celltype_L4_L5_Refine','Celltype_L4_L5_Refine_R2'],
           frameon=False,
           legend_fontsize=4, legend_fontoutline=2,
           size=4,
           )

In [212]:
groupby = "L4_leiden_TOTALVI_0.5"

In [90]:
pd.set_option('display.max_rows', 1000)
pd.set_option('display.max_columns', 1000)

In [91]:
xlsx = pd.ExcelWriter(f"{obj_path}{celltype}/{groupby}_freq_table.xlsx")
for group in ['SampleID', 'DonorID','scDblFinder.class',
              'Immune_All_Low', 'Adult_Human_Blood', 'Adult_Human_Bone_marrow',
              'AIFI_L2', 'AIFI_L3', 'predicted.celltype.l2',
              ]:
    freq_table_multi = adata.obs.groupby([groupby, group]).size()
    pd.DataFrame(freq_table_multi).to_excel(xlsx,sheet_name=group)
xlsx.close()

In [ ]:
#数据簇间信息比较
fig, axs = plt.subplots(5, 2, figsize=(18, 12),constrained_layout=True)
plt.subplots_adjust(hspace=1,wspace=1)
sc.pl.violin(adata, keys='nCount_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,0], show=False)
sc.pl.violin(adata, keys='nFeature_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,1], show=False)
sc.pl.violin(adata, keys='log10GenesPerUMI', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,0], show=False)
sc.pl.violin(adata, keys='percent_top50', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,1], show=False)

sc.pl.violin(adata, keys='percent_apop', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,0], show=False)
sc.pl.violin(adata, keys='percent_ribo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,1], show=False)
sc.pl.violin(adata, keys='percent_ieg', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,0], show=False)
sc.pl.violin(adata, keys='percent_oxphos', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,1], show=False)
sc.pl.violin(adata, keys='percent_hemo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,0], show=False)
sc.pl.violin(adata, keys='G2M.Score', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,1], show=False)

fig.tight_layout()
plt.savefig(f'{obj_path}{celltype}/{groupby}_TOTALVI_L3_metadata.png')

In [93]:
sc.tl.dendrogram(adata,groupby=groupby,use_rep='X_TOTALVI')
sc.tl.dendrogram(adt,groupby=groupby,use_rep='X_TOTALVI')

In [ ]:
#cosg差异
cosg.cosg(adata, key_added=f'cosg_{groupby}', groupby=groupby,
          mu=10,n_genes_user=100,remove_lowly_expressed=True,
         )
df_tmp = pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'])
df_tmp.to_csv(f"{obj_path}{celltype}/cosg_{groupby}.csv")
#cosg差异作图
df_tmp=pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'][:8,]).T
df_tmp=df_tmp.reindex(adata.uns['dendrogram_'+groupby]['categories_ordered'])
marker_genes_list={idx: list(row.values) for idx, row in df_tmp.iterrows()}
marker_genes_list = {k: v for k, v in marker_genes_list.items() if not any(isinstance(x, float) for x in v)}
sc.pl.dotplot(adata, marker_genes_list,
             groupby=groupby,
             dendrogram=True,
             swap_axes=False,
             standard_scale='var',
             save=f'cosg_{groupby}',
             cmap='Spectral_r')

In [ ]:
sc.pl.umap(adt, color='HLA-DR',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',layer='dsb')

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,0],show=False)
sc.pl.umap(adt, color='CD183',legend_fontsize=4, legend_fontoutline=2,legend_loc='on data',ax=axs[0,1],show=False,layer='dsb')
sc.pl.umap(adt, color='CD185',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,2],show=False,layer='dsb')
sc.pl.umap(adt, color='CD196',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,3],show=False,layer='dsb')
sc.pl.umap(adt, color='CD161',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,0],show=False,layer='dsb')
sc.pl.umap(adt, color='CD45RA',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,1],show=False,layer='dsb')
sc.pl.umap(adt, color='CD16',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,2],show=False,layer='dsb')
sc.pl.umap(adt, color='IgM',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,3],show=False,layer='dsb')
plt.show()

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,0],show=False)
sc.pl.umap(adt, color='Celltype_L4_L5_Refine_R2',legend_fontsize=4, legend_fontoutline=2,legend_loc='on data',ax=axs[0,1],show=False)
sc.pl.umap(adata, color='CCR4',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,2],show=False)
sc.pl.umap(adata, color='GZMK',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,3],show=False)
sc.pl.umap(adata, color='KLRB1',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,0],show=False)
sc.pl.umap(adata, color='IFNG',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,1],show=False)
sc.pl.umap(adata, color='CXCR3',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,2],show=False)
sc.pl.umap(adata, color='CCR6',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,3],show=False)
plt.show()

In [ ]:
sc.pl.umap(adata, color='CCR4',legend_fontsize=4, legend_fontoutline=2,legend_loc='on data',size=2)

In [ ]:
sc.pl.umap(adata, color='CCR10',legend_fontsize=4, legend_fontoutline=2,legend_loc='on data',size=2)

In [ ]:
#Th17
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3','AHR',
                      'LTK','PTPN13','PDE4D','CCR6','RORC','NR1D1','CTSH','KIF5C','LGALS3','USP10','CMTM6','TOB1',
                     'TNFSF13B','CISH','AQP3','AUTS2','NSG1','S100A4'],
              standard_scale='var',groupby=groupby)

In [ ]:
#Th1/Th17
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'GZMH','IL18RAP','S1PR5','LYAR','NKG7','CST7','PRF1','TBX21','LINC01871','KLRG1','MYBL1','EOMES',
                     'EFHD2','DUSP2','SAMD3','CTSW','ID2','MATK','HOPX',],
              standard_scale='var',groupby=groupby)

In [ ]:
#Th1
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3','TBX21','IFNG',
                      'CMC1','CST7','FCRL3','CCL4','SLAMF7','EOMES','PDCD1','NKG7','CCR5','KLRK1','F2R','PLEK'],
              standard_scale='var',groupby=groupby)

In [ ]:
#Th22
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'CRIP1','LGALS1','LGALS1','S100A10','S100A4','PI16','LMNA','ANXA5','ANXA2'],
              standard_scale='var',groupby=groupby)

In [ ]:
#Th2
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'PTGDR2','SNED1','NEFL','GATA3','FXYD7','C1orf162','GDPD5','IL4R','CAPG',
                     'LGALS1','TNFSF10','TNFRSF4','PPP1R9B','CSGALNACT1','NIBAN1','ERN1','SORL1','RUNX2'],
              standard_scale='var',groupby=groupby)

In [ ]:
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'LIMS1','RORC','CD27','CCR1','CCR2','PRDM1','CD27','CD28','CXCR5','KLRB1',
                      'TCF7','ICOS','CD27','CXCR3','PDCD1','TBX21','IL6R','TIGIT','LAG3','HAVCR2','BTBD9',
                      'CCL5','CTLA4','CD40LG'],
              standard_scale='var',groupby=groupby)

In [ ]:
#Th from BD
#TNFSF8=CD30L B3GAT1=CD57  BTLA= CD272  SLAMF5=CD84 HAVCR2=CD365
sc.pl.dotplot(adata, ['CXCR5','IL6R','TNFSF8','NRP1','IL21R','B3GAT1','BCL6','MAF','STAT3','ICOS','PDCD1','TIGIT','BTLA','CD200','SLAMF1','CD84',#Tfh
                      'IL4','IL17F','IL17A','IL21',#tfh分泌
                      'GATA3','SMAD1','STAT6','SPI1','IRF4',#Th9
                      'IL9','IL10','CCL17','CCL22','TGFB1',#th9分泌
                      'HAVCR2','CXCR4','CCR3','CCR4','CCR8','PTGDR2','GATA3','STAT5A','STAT6','MAF','GFI1','IRF4','NOTCH1','NOTCH2','IL1RL1','IL17RB','IFNGR1','IFNGR2','TNFRSF8',#Th2
                      'IL2','IL5','IL6','IL10','IL13','IL31',#Th2分泌
                      'CCR4','CCR6','CCR10','AHR','PDGFRA','PDGFRB',#Th22
                      'IL22','TNF',#Th22分泌
                      'CXCR3','CCR5','KLRD1','TBX21','STAT1','STAT4','EOMES','RUNX3','FASLG','IL12RB1','IL12RB2','IL18R1','IL27RA','NOTCH3','TNFSF11','ICOS','HAVCR2','DPP4',#Th1
                      'LTB','LTA','PRF1','GZMB','GZMA','TNF','IFNG',#Th1分泌
                      'CCR4','CCR6','KLRB1','ICOS','HAVCR2','RORC','RORA','STAT3','RUNX1','BATF','IRF4','MAF','IL6R','IL13RA1','IL21R','IL23R',#Th17
                      'TNF','CCL20','IL17A','IL17F','IL21','IL22','IL24','IL26',#Th17分泌
                      ],
              standard_scale='var',groupby=groupby)

In [ ]:
adata.obs['cluster_dummy']="NOT"
adata.obs.loc[adata.obs[groupby]=="6",'cluster_dummy'] = "YES"
sc.pl.umap(adata, color='cluster_dummy')

In [213]:
adata.obs[groupby].value_counts()

L4_leiden_TOTALVI_0.5
0    497199
2    326494
1    271745
3    206814
Name: count, dtype: int64

In [214]:
#Naïve CD4+ T 需要CD45RA+ CCR7=CD197hi CD62Lhi
# effector memory T cells(TEM, CD45RA-/CCR7-)
# TEMRA cells, which are T cells that re-express CD45RA(CD45RA+/CCR7-)

#'Th22':#CCR4+ CCR6+ CCR10+ KLRB1- GZMK- GATA3lo CCL5- GATA3+,CCR4+ CCR6+ KLRB1- GZMKlo AHRlo
#'Th1/Th17':# CCR6+ CCR4- CXCR3+ KLRB1+ GZMK+ CCL5+
#'Th17':#step1 CCR6+ CCR4- KLRB1+ GZMK- CCL5- RORC+ GZMK- TBX21-
#'Th2':#step1 CCR4+ CCR6- KLRB1- GATA3+ GZMK- CCL5-；CCR4+ CXCR5- GATA3+ CCL5-.CD45RA- CD279- TBX21- LEF1+ ,且没有CCL5- GZMH- GZMK- GZMB- KLRB1-各种标记的情况下CD62L+ CD27+ CD25+ 
#'Th1':#step3 确认 CXCR3+ CCR6- KLRB1- GZMK+ CD279+ GZMK+ CCL5+. CD279+(核心) GZMK+(核心) TBX21+ CCL5+(核心) TIGIT+ KLRB1-(核心) CD127-/IL7Rlo CCR7-CD197- SELLlo/CD62Llo  GZMH- GNLY- PRF1- GZMB- 的是Th1

cell_dict = {#'Th22':['',],#CCR4+ CCR6+ CCR10+ KLRB1lo/- GZMK-    CRIP1 LGALS1 LGALS3 PI16 ANXA5 ANXA2
             'Th2|Th22':['2'],#PTGDR2=CRTH2 GATA3++ IL4R+ CCR4+ CCR6- KLRB1- GZMK- CCL5- CD62L+ CD25+     PTGDR2,SNED1,NEFL,GATA3,FXYD7,C1orf162
             'Th1':['3',],#EOMES+ GZMK+ CCL5+ KLRB1- CXCR3+ CCR6- CD279+ TBX21+ IFNG+.  CMC1,CST7,FCRL3,CCL4,SLAMF7,EOMES,PDCD1,NKG7,CCR5,KLRK1,F2R,PLEK
             'Th17':['0'],#CCR6=CD196++ RORC+ CCR4- KLRB1+ GZMK- CCL5- ICOS+    高表达LTK,PTPN13,PDE4D,CCR6,RORC,NR1D1,CTSH,KIF5C,LGALS3,USP10,CMTM6,TOB1
             'Th1/Th17':['1',],#DPP4+ CCR6+ EOMESlo CCR6lo CCR4- CXCR3+ KLRB1+ GZMKlo/+ CCL5+ TBX21+ 与Th1相似, 差异基因中等表达
            }

In [215]:
check_dict_duplicates(cell_dict)

'无重复'

In [216]:
# Generate new assignments
for i in cell_dict.keys():
    ind = pd.Series(adata.obs[groupby]).isin(cell_dict[i])
    adata.obs.loc[ind,'Celltype_L4_L5_Refine_R3'] = i

In [217]:
(adata.obs['Celltype_L4_L5_Refine_R3'].isna()).value_counts()

Celltype_L4_L5_Refine_R3
False    1302252
Name: count, dtype: int64

In [218]:
adata.obs['Celltype_L4_L5_Refine_R3'].value_counts()

Celltype_L4_L5_Refine_R3
Th17        497199
Th2|Th22    326494
Th1/Th17    271745
Th1         206814
Name: count, dtype: int64

In [219]:
adata.obs['L4_leiden_TOTALVI_0.8'].value_counts()

L4_leiden_TOTALVI_0.8
0    376819
3    341926
4    299904
2    175745
1    107768
5        90
Name: count, dtype: int64

In [220]:
adata = adata[adata.obs['L4_leiden_TOTALVI_0.8'] != "5",:]# CD4 CTL

In [221]:
adata.obs['Celltype_L4_L5_Refine_R3'].value_counts()

Celltype_L4_L5_Refine_R3
Th17        497199
Th2|Th22    326494
Th1/Th17    271661
Th1         206808
Name: count, dtype: int64

In [222]:
indices = adata.obs.loc[:,['Celltype_L1_L2','Celltype_L1_L2_Refine','Celltype_L2_L3_Refine','Celltype_L3_L4_Refine','Celltype_L4_L5_Refine','Celltype_L4_L5_Refine_R2','Celltype_L4_L5_Refine_R3','receptor_type','receptor_type_BCR']]
indices.to_csv(f"{obj_path}/{celltype}/R6_refine_indices_{celltype}.csv")

# 4. 定 终 Tfh_Tcm 作为不同的分类体系，需要放入最终的参考系中，比较后决定。后续有新细胞纳入

In [149]:
celltype="Tfh_Tcm"
R_data_path = f"{obj_path}{celltype}"
sc.settings.figdir=f"{obj_path}{celltype}"

In [150]:
adata,adt,leiden_data = get_norm_annotation_data(celltype)

In [ ]:
sc.pl.umap(adata, color=['Celltype_L4_L5_Refine','Celltype_L4_L5_Refine_R2'],
           frameon=False,
           legend_fontsize=4, legend_fontoutline=2,
           size=4,
           )

In [151]:
groupby = "L4_leiden_TOTALVI_0.8"

In [ ]:
from sklearn_ann.kneighbors.annoy import AnnoyTransformer

In [ ]:
sc.pp.neighbors(adata, transformer=AnnoyTransformer(20), use_rep='X_TOTALVI')

In [ ]:
sc.tl.leiden(adata, key_added=f'L4_leiden_TOTALVI_2', 
                 resolution=2,
                 use_weights=True,directed=False,flavor="igraph")

In [ ]:
sc.pl.umap(adata, color='L4_leiden_TOTALVI_2',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data')

In [ ]:
pd.set_option('display.max_rows', 1000)
pd.set_option('display.max_columns', 1000)

In [ ]:
xlsx = pd.ExcelWriter(f"{obj_path}{celltype}/{groupby}_freq_table.xlsx")
for group in ['SampleID', 'DonorID','scDblFinder.class',
              'Immune_All_Low', 'Adult_Human_Blood', 'Adult_Human_Bone_marrow',
              'AIFI_L2', 'AIFI_L3', 'predicted.celltype.l2',
              ]:
    freq_table_multi = adata.obs.groupby([groupby, group]).size()
    pd.DataFrame(freq_table_multi).to_excel(xlsx,sheet_name=group)
xlsx.close()

In [ ]:
#数据簇间信息比较
fig, axs = plt.subplots(5, 2, figsize=(18, 12),constrained_layout=True)
plt.subplots_adjust(hspace=1,wspace=1)
sc.pl.violin(adata, keys='nCount_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,0], show=False)
sc.pl.violin(adata, keys='nFeature_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,1], show=False)
sc.pl.violin(adata, keys='log10GenesPerUMI', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,0], show=False)
sc.pl.violin(adata, keys='percent_top50', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,1], show=False)

sc.pl.violin(adata, keys='percent_apop', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,0], show=False)
sc.pl.violin(adata, keys='percent_ribo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,1], show=False)
sc.pl.violin(adata, keys='percent_ieg', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,0], show=False)
sc.pl.violin(adata, keys='percent_oxphos', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,1], show=False)
sc.pl.violin(adata, keys='percent_hemo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,0], show=False)
sc.pl.violin(adata, keys='G2M.Score', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,1], show=False)

fig.tight_layout()
plt.savefig(f'{obj_path}{celltype}/{groupby}_TOTALVI_L3_metadata.png')

In [183]:
sc.tl.dendrogram(adata,groupby=groupby,use_rep='X_TOTALVI')
sc.tl.dendrogram(adt,groupby=groupby,use_rep='X_TOTALVI')

In [ ]:
#cosg差异
cosg.cosg(adata, key_added=f'cosg_{groupby}', groupby=groupby,
          mu=10,n_genes_user=100,remove_lowly_expressed=True,
         )
df_tmp = pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'])
df_tmp.to_csv(f"{obj_path}{celltype}/cosg_{groupby}.csv")
#cosg差异作图
df_tmp=pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'][:8,]).T
df_tmp=df_tmp.reindex(adata.uns['dendrogram_'+groupby]['categories_ordered'])
marker_genes_list={idx: list(row.values) for idx, row in df_tmp.iterrows()}
marker_genes_list = {k: v for k, v in marker_genes_list.items() if not any(isinstance(x, float) for x in v)}
sc.pl.dotplot(adata, marker_genes_list,
             groupby=groupby,
             dendrogram=True,
             swap_axes=False,
             standard_scale='var',
             save=f'cosg_{groupby}',
             cmap='Spectral_r')

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,0],show=False)
sc.pl.umap(adt, color='CD183',legend_fontsize=4, legend_fontoutline=2,legend_loc='on data',ax=axs[0,1],show=False,layer='dsb')
sc.pl.umap(adt, color='CD185',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,2],show=False,layer='dsb')
sc.pl.umap(adt, color='CD196',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,3],show=False,layer='dsb')
sc.pl.umap(adt, color='CD161',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,0],show=False,layer='dsb')
sc.pl.umap(adt, color='CD45RA',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,1],show=False,layer='dsb')
sc.pl.umap(adt, color='CD16',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,2],show=False,layer='dsb')
sc.pl.umap(adt, color='CD197',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,3],show=False,layer='dsb')
plt.show()

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,0],show=False)
sc.pl.umap(adt, color='CD45RA',legend_fontsize=4, legend_fontoutline=2,legend_loc='on data',ax=axs[0,1],show=False)
sc.pl.umap(adata, color='CCL5',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,2],show=False)
sc.pl.umap(adata, color='CCR5',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,3],show=False)
sc.pl.umap(adata, color='KLRB1',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,0],show=False)
sc.pl.umap(adata, color='CCR4',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,1],show=False)
sc.pl.umap(adata, color='CXCR5',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,2],show=False)
sc.pl.umap(adata, color='CCR7',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,3],show=False)
plt.show()

In [ ]:
sc.pl.umap(adata, color='GZMH',legend_fontsize=4, legend_fontoutline=2,legend_loc='on data')

In [ ]:
fig, axs = plt.subplots(ncols=2, nrows=1, figsize=(16, 8))
sc.pl.umap(adt, color='CD27',legend_fontsize=4, legend_fontoutline=2,legend_loc='on data',ax=axs[0],show=False,layer='dsb')
sc.pl.umap(adt, color='CD28',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1],show=False,layer='dsb')
plt.show()

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,0],show=False)
sc.pl.umap(adt, color='CD185',legend_fontsize=4, legend_fontoutline=2,legend_loc='on data',ax=axs[0,1],show=False)
sc.pl.umap(adata, color='CCR4',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,2],show=False)
sc.pl.umap(adata, color='GZMK',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,3],show=False)
sc.pl.umap(adata, color='KLRB1',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,0],show=False)
sc.pl.umap(adata, color='CXCR5',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,1],show=False)
sc.pl.umap(adata, color='CXCR3',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,2],show=False)
sc.pl.umap(adata, color='CCR6',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,3],show=False)
plt.show()

In [ ]:
sc.pl.umap(adt, color='CD196',legend_fontsize=4, legend_fontoutline=2,legend_loc='on data',size=2,layer='dsb')

In [ ]:
#Th17
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3','AHR',
                      'LTK','PTPN13','PDE4D','CCR6','RORC','NR1D1','CTSH','KIF5C','LGALS3','USP10','CMTM6','TOB1',
                     'TNFSF13B','CISH','AQP3','AUTS2','NSG1','S100A4'],
              standard_scale='obs',groupby=groupby)

In [ ]:
#Th1/Th17
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'GZMH','IL18RAP','S1PR5','LYAR','NKG7','CST7','PRF1','TBX21','LINC01871','KLRG1','MYBL1','EOMES',
                     'EFHD2','DUSP2','SAMD3','CTSW','ID2','MATK','HOPX',],
              standard_scale='obs',groupby=groupby)

In [ ]:
#Th1
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3','TBX21','IFNG',
                      'CMC1','CST7','FCRL3','CCL4','SLAMF7','EOMES','PDCD1','NKG7','CCR5','KLRK1','F2R','PLEK'],
              standard_scale='obs',groupby=groupby)

In [ ]:
#Th22
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'CRIP1','LGALS1','LGALS1','S100A10','S100A4','PI16','LMNA','ANXA5','ANXA2'],
              standard_scale='obs',groupby=groupby)

In [ ]:
#Th2
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'PTGDR2','SNED1','NEFL','GATA3','FXYD7','C1orf162','GDPD5','IL4R','CAPG',
                     'LGALS1','TNFSF10','TNFRSF4','PPP1R9B','CSGALNACT1','NIBAN1','ERN1','SORL1','RUNX2'],
              standard_scale='obs',groupby=groupby)

In [ ]:
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'LIMS1','RORC','CD27','CCR1','CCR2','PRDM1','CD27','CD28','CXCR5','KLRB1',
                      'TCF7','ICOS','CD27','CXCR3','PDCD1','TBX21','IL6R','TIGIT','LAG3','HAVCR2','BTBD9',
                      'CCL5','CTLA4','CD40LG'],
              standard_scale='obs',groupby=groupby)

In [ ]:
#Th from BD
#TNFSF8=CD30L B3GAT1=CD57  BTLA= CD272  SLAMF5=CD84 HAVCR2=CD365
sc.pl.dotplot(adata, ['CXCR5','IL6R','TNFSF8','NRP1','IL21R','B3GAT1','BCL6','MAF','STAT3','ICOS','PDCD1','TIGIT','BTLA','CD200','SLAMF1','CD84',#Tfh
                      'IL4','IL17F','IL17A','IL21',#tfh分泌
                      'GATA3','SMAD1','STAT6','SPI1','IRF4',#Th9
                      'IL9','IL10','CCL17','CCL22','TGFB1',#th9分泌
                      'HAVCR2','CXCR4','CCR3','CCR4','CCR8','PTGDR2','GATA3','STAT5A','STAT6','MAF','GFI1','IRF4','NOTCH1','NOTCH2','IL1RL1','IL17RB','IFNGR1','IFNGR2','TNFRSF8',#Th2
                      'IL2','IL5','IL6','IL10','IL13','IL31',#Th2分泌
                      'CCR4','CCR6','CCR10','AHR','PDGFRA','PDGFRB',#Th22
                      'IL22','TNF',#Th22分泌
                      'CXCR3','CCR5','KLRD1','TBX21','STAT1','STAT4','EOMES','RUNX3','FASLG','IL12RB1','IL12RB2','IL18R1','IL27RA','NOTCH3','TNFSF11','ICOS','HAVCR2','DPP4',#Th1
                      'LTB','LTA','PRF1','GZMB','GZMA','TNF','IFNG',#Th1分泌
                      'CCR4','CCR6','KLRB1','ICOS','HAVCR2','RORC','RORA','STAT3','RUNX1','BATF','IRF4','MAF','IL6R','IL13RA1','IL21R','IL23R',#Th17
                      'TNF','CCL20','IL17A','IL17F','IL21','IL22','IL24','IL26',#Th17分泌
                      ],
              standard_scale='obs',groupby=groupby)

In [ ]:
adata.obs['cluster_dummy']="NOT"
adata.obs.loc[adata.obs[groupby]=="1",'cluster_dummy'] = "YES"
sc.pl.umap(adata, color='cluster_dummy')

In [152]:
adata.obs[groupby].value_counts()

L4_leiden_TOTALVI_0.8
2    401373
3    383434
0    232117
1    108341
5       832
7        13
4         6
6         6
9         5
8         3
Name: count, dtype: int64

In [153]:
#Naïve CD4+ T 需要CD45RA+ CCR7=CD197hi CD62Lhi
# effector memory T cells(TEM, CD45RA-/CCR7-)
# TEMRA cells, which are T cells that re-express CD45RA(CD45RA+/CCR7-)

#'Th22':#CCR4+ CCR6+ CCR10+ KLRB1- GZMK- GATA3lo CCL5- GATA3+,CCR4+ CCR6+ KLRB1- GZMKlo AHRlo
#'Th1/Th17':# CCR6+ CCR4- CXCR3+ KLRB1+ GZMK+ CCL5+
#'Th17':#step1 CCR6+ CCR4- KLRB1+ GZMK- CCL5- RORC+ GZMK- TBX21-
#'Th2':#step1 CCR4+ CCR6- KLRB1- GATA3+ GZMK- CCL5-；CCR4+ CXCR5- GATA3+ CCL5-.CD45RA- CD279- TBX21- LEF1+ ,且没有CCL5- GZMH- GZMK- GZMB- KLRB1-各种标记的情况下CD62L+ CD27+ CD25+ 
#'Th1':#step3 确认 CXCR3+ CCR6- KLRB1- GZMK+ CD279+ GZMK+ CCL5+. CD279+(核心) GZMK+(核心) TBX21+ CCL5+(核心) TIGIT+ KLRB1-(核心) CD127-/IL7Rlo CCR7-CD197- SELLlo/CD62Llo  GZMH- GNLY- PRF1- GZMB- 的是Th1

cell_dict = {'Tfh':['0','1','2','3',],#IL7R TCF7 CD27 ITGB1 。高表达CD27,用于形成记忆，不表达CCR7等,表达SELL
             'Doublet|Lowquality':['5',#Naive
                                   '4','6','7','8','9',#lncRNA expression and few cells
                         ],#
            }

In [154]:
check_dict_duplicates(cell_dict)

'无重复'

In [155]:
# Generate new assignments
for i in cell_dict.keys():
    ind = pd.Series(adata.obs[groupby]).isin(cell_dict[i])
    adata.obs.loc[ind,'Celltype_L4_L5_Refine_R3'] = i

In [156]:
(adata.obs['Celltype_L4_L5_Refine_R3'].isna()).value_counts()

Celltype_L4_L5_Refine_R3
False    1126130
Name: count, dtype: int64

In [157]:
adata.obs['Celltype_L4_L5_Refine_R3'].value_counts()

Celltype_L4_L5_Refine_R3
Tfh                   1125265
Doublet|Lowquality        865
Name: count, dtype: int64

In [158]:
adata = adata[adata.obs['Celltype_L4_L5_Refine_R3'] != "Doublet|Lowquality",:]

In [159]:
adata.obs['Celltype_L4_L5_Refine_R3'].value_counts()

Celltype_L4_L5_Refine_R3
Tfh    1125265
Name: count, dtype: int64

In [160]:
indices = adata.obs.loc[:,['Celltype_L1_L2','Celltype_L1_L2_Refine','Celltype_L2_L3_Refine',
                           'Celltype_L3_L4_Refine','Celltype_L4_L5_Refine',
                           'Celltype_L4_L5_Refine_R2','Celltype_L4_L5_Refine_R3',
                           groupby,'receptor_type','receptor_type_BCR']]
indices['UMAP_1'] = adata.obsm['X_umap'][:, 0].copy()
indices['UMAP_2'] = adata.obsm['X_umap'][:, 1].copy()
indices.rename(columns={groupby: 'leiden_cluster'}, inplace=True) #Save the cluster categorical, check the relationship between clusters and clinical to avoid missing someone.
indices['leiden_cluster'] = celltype + " c" + indices['leiden_cluster'].astype(str)
os.makedirs(f'{finnal_path}/{celltype}', exist_ok=True)
indices.to_csv(f"{finnal_path}/{celltype}/Finnal_indices_{celltype}.csv")
indices.head()

,Celltype_L1_L2,Celltype_L1_L2_Refine,Celltype_L2_L3_Refine,Celltype_L3_L4_Refine,Celltype_L4_L5_Refine,Celltype_L4_L5_Refine_R2,Celltype_L4_L5_Refine_R3,leiden_cluster,receptor_type,receptor_type_BCR,UMAP_1,UMAP_2
D0589_Rep1_CATACCAA_ATCATTCC_AATGTTGC,CD4+ T,Naïve CD4+ T,Treg,Tem CD4+ T(Treg),T help Memory(fromTreg),Tfh/Tcm,Tfh,Tfh_Tcm c0,TCR,NaN,-0.175477,7.222757
D0589_Rep1_CTGAGCCA_AAACATCG_CGACACAC,CD4+ T,Naïve CD4+ T,Treg,Tem CD4+ T(Treg),T help Memory(fromTreg),Tfh/Tcm,Tfh,Tfh_Tcm c0,TCR,NaN,1.046298,6.978236
D0589_Rep1_CTAAGGTC_CCTCCTGA_ACCTCCAA,CD4+ T,Naïve CD4+ T,Treg,Tem CD4+ T(Treg),T help Memory(fromTreg),Tfh/Tcm,Tfh,Tfh_Tcm c2,NaN,NaN,-1.210865,7.891342
D0589_Rep1_AACGCTTA_TCCGTCTA_CACTTCGA,CD4+ T,Naïve CD4+ T,Treg,Tem CD4+ T(Treg),T help Memory(fromTreg),Tfh/Tcm,Tfh,Tfh_Tcm c0,TCR,NaN,-0.494080,7.181665
D0589_Rep1_TGGTGGTA_CGACACAC_GCTAACGA,CD4+ T,Naïve CD4+ T,Treg,Tem CD4+ T(Treg),T help Memory(fromTreg),Tfh/Tcm,Tfh,Tfh_Tcm c3,TCR,NaN,0.639964,8.822940


# 5. 定 终 Cytotoxic CD4+ T Cell Refine

In [161]:
celltype="CytotoxicCD4"
R_data_path = f"{obj_path}{celltype}"
sc.settings.figdir=f"{obj_path}{celltype}"

In [162]:
adata,adt,leiden_data = get_norm_annotation_data(celltype)

In [ ]:
sc.pl.umap(adata, color=['Celltype_L1_L2_Refine','Celltype_L4_L5_Refine','Celltype_L4_L5_Refine_R2','receptor_type'],
           frameon=False,
           legend_fontsize=4, legend_fontoutline=2,
           size=4)

In [163]:
groupby = "L4_leiden_TOTALVI_0.3"

In [114]:
xlsx = pd.ExcelWriter(f"{obj_path}{celltype}/{groupby}_freq_table.xlsx")
for group in ['SampleID', 'DonorID','scDblFinder.class',
              'Immune_All_Low', 'Adult_Human_Blood', 'Adult_Human_Bone_marrow',
              'AIFI_L2', 'AIFI_L3', 'predicted.celltype.l2',
              
              ]:
    freq_table_multi = adata.obs.groupby([groupby, group]).size()
    pd.DataFrame(freq_table_multi).to_excel(xlsx,sheet_name=group)
xlsx.close()

In [ ]:
#数据簇间信息比较
fig, axs = plt.subplots(5, 2, figsize=(18, 12),constrained_layout=True)
plt.subplots_adjust(hspace=1,wspace=1)
sc.pl.violin(adata, keys='nCount_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,0], show=False)
sc.pl.violin(adata, keys='nFeature_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,1], show=False)
sc.pl.violin(adata, keys='log10GenesPerUMI', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,0], show=False)
sc.pl.violin(adata, keys='percent_top50', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,1], show=False)

sc.pl.violin(adata, keys='percent_apop', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,0], show=False)
sc.pl.violin(adata, keys='percent_ribo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,1], show=False)
sc.pl.violin(adata, keys='percent_ieg', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,0], show=False)
sc.pl.violin(adata, keys='percent_oxphos', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,1], show=False)
sc.pl.violin(adata, keys='percent_hemo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,0], show=False)
sc.pl.violin(adata, keys='G2M.Score', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,1], show=False)

fig.tight_layout()
plt.savefig(f'{obj_path}{celltype}/{groupby}_TOTALVI_L3_metadata.png')

In [116]:
sc.tl.dendrogram(adata,groupby=groupby,use_rep='X_TOTALVI')
sc.tl.dendrogram(adt,groupby=groupby,use_rep='X_TOTALVI')

In [ ]:
#cosg差异
cosg.cosg(adata, key_added=f'cosg_{groupby}', groupby=groupby,
          mu=10,n_genes_user=100,remove_lowly_expressed=True,
         )
df_tmp = pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'])
df_tmp.to_csv(f"{obj_path}{celltype}/cosg_{groupby}.csv")
#cosg差异作图
df_tmp=pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'][:8,]).T
df_tmp=df_tmp.reindex(adata.uns['dendrogram_'+groupby]['categories_ordered'])
marker_genes_list={idx: list(row.values) for idx, row in df_tmp.iterrows()}
marker_genes_list = {k: v for k, v in marker_genes_list.items() if not any(isinstance(x, float) for x in v)}
sc.pl.dotplot(adata, marker_genes_list,
             groupby=groupby,
             dendrogram=True,
             swap_axes=False,
             standard_scale='var',
             save=f'cosg_{groupby}',
             cmap='Spectral_r')

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,0],show=False)
sc.pl.umap(adata, color='GZMB',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,1],show=False)
sc.pl.umap(adata, color='CCR4',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,2],show=False)
sc.pl.umap(adata, color='KLRB1',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,3],show=False)
sc.pl.umap(adata, color='CTLA4',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,0],show=False)
sc.pl.umap(adt, color='CD45RA',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,1],show=False,layer='clr')
sc.pl.umap(adata, color='GZMK',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,2],show=False)
sc.pl.umap(adata, color='CCR6',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,3],show=False)
plt.show()

In [ ]:
sc.pl.umap(adata, color='PDCD1',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data')

In [ ]:
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMB','BTBD9','GATA3',
                      'LIMS1','RORC','CD27','CCR1','CCR2','PRDM1','CD27','CD28','CXCR5','KLRB1',
                      'TCF7','ICOS','CD27','CXCR3','PDCD1','TBX21','IL6R','TIGIT','HAVCR2','LAG3',
                      'CCL5','CTLA4','CD40LG','CRTAM'],
              standard_scale='obs',groupby=groupby)

In [121]:
adata.obs['Celltype_L4_L5_Refine_R2'].value_counts()

Celltype_L4_L5_Refine_R2
Terminal effector CD4+ T             686211
Terminal effector HLA-DRhi CD4+ T     30807
Temra CD4+ T                          27780
Name: count, dtype: int64

In [164]:
adata.obs[groupby].value_counts()

L4_leiden_TOTALVI_0.3
1    437315
0    213387
2     66915
4     14511
3     12670
Name: count, dtype: int64

In [165]:
#Naïve CD4+ T 需要CD45RA+ CCR7=CD197hi CD62Lhi
# effector memory T cells(TEM, CD45RA-/CCR7-)
# TEMRA cells, which are T cells that re-express CD45RA(CD45RA+/CCR7-)


cell_dict = {'GZMB+ CD4+ terminal effector T cells':['0','1','2'],#
             'CD4+ Temra':['4'],#
             'HLA-DRhi CD4+ terminal effector T cells':['3'],#
            }

In [166]:
check_dict_duplicates(cell_dict)

'无重复'

In [167]:
# Generate new assignments
for i in cell_dict.keys():
    ind = pd.Series(adata.obs[groupby]).isin(cell_dict[i])
    adata.obs.loc[ind,'Celltype_L4_L5_Refine_R3'] = i

In [168]:
(adata.obs['Celltype_L4_L5_Refine_R3'].isna()).value_counts()

Celltype_L4_L5_Refine_R3
False    744798
Name: count, dtype: int64

In [169]:
adata.obs['Celltype_L4_L5_Refine_R3'].value_counts()

Celltype_L4_L5_Refine_R3
GZMB+ CD4+ terminal effector T cells       717617
CD4+ Temra                                  14511
HLA-DRhi CD4+ terminal effector T cells     12670
Name: count, dtype: int64

In [170]:
indices = adata.obs.loc[:,['Celltype_L1_L2','Celltype_L1_L2_Refine','Celltype_L2_L3_Refine',
                           'Celltype_L3_L4_Refine','Celltype_L4_L5_Refine',
                           'Celltype_L4_L5_Refine_R2','Celltype_L4_L5_Refine_R3',
                           groupby,'receptor_type','receptor_type_BCR']]
indices['UMAP_1'] = adata.obsm['X_umap'][:, 0].copy()
indices['UMAP_2'] = adata.obsm['X_umap'][:, 1].copy()
indices.rename(columns={groupby: 'leiden_cluster'}, inplace=True) #Save the cluster categorical, check the relationship between clusters and clinical to avoid missing someone.
indices['leiden_cluster'] = celltype + " c" + indices['leiden_cluster'].astype(str)
os.makedirs(f'{finnal_path}/{celltype}', exist_ok=True)
indices.to_csv(f"{finnal_path}/{celltype}/Finnal_indices_{celltype}.csv")
indices.head()

,Celltype_L1_L2,Celltype_L1_L2_Refine,Celltype_L2_L3_Refine,Celltype_L3_L4_Refine,Celltype_L4_L5_Refine,Celltype_L4_L5_Refine_R2,Celltype_L4_L5_Refine_R3,leiden_cluster,receptor_type,receptor_type_BCR,UMAP_1,UMAP_2
D0589_Rep1_ATTGAGGA_GGAGAACA_AGTGGTCA,CD4+ T,Cytotoxic CD4+ T,Cytotoxic CD4+ T,Cytotoxic CD4+ T,Cytotoxic CD4+ T,Terminal effector CD4+ T,GZMB+ CD4+ terminal effector T cells,CytotoxicCD4 c0,TCR,NaN,-3.565122,1.723382
D0589_Rep1_CCGTGAGA_AAGGACAC_GAACAGGC,CD4+ T,Cytotoxic CD4+ T,Cytotoxic CD4+ T,Cytotoxic CD4+ T,Cytotoxic CD4+ T,Terminal effector CD4+ T,GZMB+ CD4+ terminal effector T cells,CytotoxicCD4 c1,TCR,NaN,-1.376033,3.284809
D0589_Rep1_CTAAGGTC_TGGAACAA_ACGTATCA,CD4+ T,Cytotoxic CD4+ T,Cytotoxic CD4+ T,Cytotoxic CD4+ T,Cytotoxic CD4+ T,Terminal effector CD4+ T,GZMB+ CD4+ terminal effector T cells,CytotoxicCD4 c1,TCR,NaN,-2.082267,0.852539
D0589_Rep1_CGGATTGC_AACGTGAT_AGGCTAAC,CD4+ T,Cytotoxic CD4+ T,Cytotoxic CD4+ T,Cytotoxic CD4+ T,Cytotoxic CD4+ T,Terminal effector CD4+ T,GZMB+ CD4+ terminal effector T cells,CytotoxicCD4 c0,TCR,NaN,-2.380426,4.618530
D0589_Rep1_CAAGGAGC_ACAGCAGA_AATCCGTC,CD4+ T,Cytotoxic CD4+ T,Cytotoxic CD4+ T,Cytotoxic CD4+ T,Cytotoxic CD4+ T,Terminal effector CD4+ T,GZMB+ CD4+ terminal effector T cells,CytotoxicCD4 c0,TCR,NaN,-3.294155,3.508850


# 6. 定 终 MAIT Cell Refine

In [171]:
celltype="MAIT"
R_data_path = f"{obj_path}{celltype}"
sc.settings.figdir=f"{obj_path}{celltype}"

In [172]:
adata,adt,leiden_data = get_norm_annotation_data(celltype)

In [19]:
adata.obs['Celltype_L4_L5_Refine'].value_counts()

Celltype_L4_L5_Refine
MAIT    284925
Name: count, dtype: int64

In [ ]:
sc.pl.umap(adata, color=['Celltype_L1_L2_Refine','Celltype_L4_L5_Refine'],
           frameon=False,
           legend_fontsize=4, legend_fontoutline=2,
           size=4)

In [173]:
groupby = "L4_leiden_TOTALVI_0.8"

In [41]:
xlsx = pd.ExcelWriter(f"{obj_path}{celltype}/{groupby}_freq_table.xlsx")
for group in ['SampleID', 'DonorID','scDblFinder.class',
              'Immune_All_Low', 'Adult_Human_Blood', 'Adult_Human_Bone_marrow',
              'AIFI_L2', 'AIFI_L3', 'predicted.celltype.l2',
              
              ]:
    freq_table_multi = adata.obs.groupby([groupby, group]).size()
    pd.DataFrame(freq_table_multi).to_excel(xlsx,sheet_name=group)
xlsx.close()

In [ ]:
#数据簇间信息比较
fig, axs = plt.subplots(5, 2, figsize=(18, 12),constrained_layout=True)
plt.subplots_adjust(hspace=1,wspace=1)
sc.pl.violin(adata, keys='nCount_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,0], show=False)
sc.pl.violin(adata, keys='nFeature_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,1], show=False)
sc.pl.violin(adata, keys='log10GenesPerUMI', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,0], show=False)
sc.pl.violin(adata, keys='percent_top50', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,1], show=False)

sc.pl.violin(adata, keys='percent_apop', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,0], show=False)
sc.pl.violin(adata, keys='percent_ribo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,1], show=False)
sc.pl.violin(adata, keys='percent_ieg', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,0], show=False)
sc.pl.violin(adata, keys='percent_oxphos', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,1], show=False)
sc.pl.violin(adata, keys='percent_hemo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,0], show=False)
sc.pl.violin(adata, keys='G2M.Score', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,1], show=False)

fig.tight_layout()
plt.savefig(f'{obj_path}{celltype}/{groupby}_TOTALVI_L3_metadata.png')

In [43]:
sc.tl.dendrogram(adata,groupby=groupby,use_rep='X_TOTALVI')
sc.tl.dendrogram(adt,groupby=groupby,use_rep='X_TOTALVI')

In [ ]:
#cosg差异
cosg.cosg(adata, key_added=f'cosg_{groupby}', groupby=groupby,
          mu=10,n_genes_user=100,remove_lowly_expressed=True,
         )
df_tmp = pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'])
df_tmp.to_csv(f"{obj_path}{celltype}/cosg_{groupby}.csv")
#cosg差异作图
df_tmp=pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'][:8,]).T
df_tmp=df_tmp.reindex(adata.uns['dendrogram_'+groupby]['categories_ordered'])
marker_genes_list={idx: list(row.values) for idx, row in df_tmp.iterrows()}
marker_genes_list = {k: v for k, v in marker_genes_list.items() if not any(isinstance(x, float) for x in v)}
sc.pl.dotplot(adata, marker_genes_list,
             groupby=groupby,
             dendrogram=True,
             swap_axes=False,
             standard_scale='var',
             save=f'cosg_{groupby}',
             cmap='Spectral_r')

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,0],show=False)
sc.pl.umap(adata, color='TRAV1-2',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,1],show=False)
sc.pl.umap(adata, color='CD27',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,2],show=False)
sc.pl.umap(adt, color='CD27',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,3],show=False,layer='clr')
#sc.pl.violin(adt,'CD45RA',groupby=groupby,show=False, ax=axs[0,3],size=0.5)
#TNFAIP3/CXCR4/DDIT4/FOSL2/IRS2/BTG1/ZFP36/PNRC1/CDKN1B/PLIN2/BHLHE40/PLAUR/ETS1/DUSP1
#SELL/TXNIP/IFITM2/B2M/HLA-C

sc.pl.violin(adata,'CD3D',groupby=groupby,show=False, ax=axs[1,0],size=0.5)
sc.pl.violin(adata,'KLRB1',groupby=groupby,show=False, ax=axs[1,1],size=0.5)
sc.pl.violin(adata,'TRAV1-2',groupby=groupby,show=False, ax=axs[1,2],size=0.5)
sc.pl.violin(adata,'SLC4A10',groupby=groupby,show=False, ax=axs[1,3],size=0.5)
plt.show()

In [174]:
adata.obs[groupby].value_counts()

L4_leiden_TOTALVI_0.8
7     49235
2     38120
1     36044
4     34101
0     31649
3     27589
9     23087
5     18025
6     14120
10     7516
8      4684
11      755
Name: count, dtype: int64

In [ ]:
sc.pl.dotplot(adata, ['ISG15','JUN','FOS','IL32','TRAV1-2','SLC4A10','KLRB1','S100A4','LTB','GNLY','KLRD1','TRDV2','TRGV9'],
              standard_scale='obs',groupby=groupby,dendrogram=True)

In [ ]:
#CD185 (CXCR5) CD183 (CXCR3) CD278 (ICOS) CD279 (PD1)
sc.pl.dotplot(adt, ['CD45RA','CD197','CD62L','CD161','CD279','CD185','CD183','CD278','CD279','CD27','CD25'],
              standard_scale='obs',groupby=groupby)

In [ ]:
#Th17
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'LTK','PTPN13','PDE4D','CCR6','RORC','NR1D1','CTSH','KIF5C','LGALS3','USP10','CMTM6','TOB1',
                     'TNFSF13B','CISH','AQP3','AUTS2','NSG1','S100A4'],
              standard_scale='var',groupby=groupby)

In [ ]:
#Th1/Th17
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'GZMH','IL18RAP','S1PR5','LYAR','NKG7','CST7','PRF1','TBX21','LINC01871','KLRG1','MYBL1','EOMES',
                     'EFHD2','DUSP2','SAMD3','CTSW','ID2','MATK','HOPX',],
              standard_scale='var',groupby=groupby)

In [ ]:
#Th1
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'CMC1','CST7','FCRL3','CCL4','SLAMF7','EOMES','PDCD1','NKG7','CCR5','KLRK1','F2R','PLEK'],
              standard_scale='var',groupby=groupby)

# 'Th22'#CCR4+ CCR6+ CCR10+ KLRB1lo/- GZMK-    CRIP1 LGALS1 LGALS3 PI16 ANXA5 ANXA2
# 'Th2'#PTGDR2=CRTH2 GATA3++ CCR4+ CCR6- KLRB1- GZMK- CCL5- CD62L+ CD25+     PTGDR2,SNED1,NEFL,GATA3,FXYD7,C1orf162
# 'Th1'#EOMES+ GZMK+ CCL5+ KLRB1- CXCR3+ CCR6- CD279+ TBX21+   CMC1,CST7,FCRL3,CCL4,SLAMF7,EOMES,PDCD1,NKG7,CCR5,KLRK1,F2R,PLEK
# 'Th17'#CCR6=CD196++ RORC+ CCR4- KLRB1+ GZMK- CCL5-    高表达LTK,PTPN13,PDE4D,CCR6,RORC,NR1D1,CTSH,KIF5C,LGALS3,USP10,CMTM6,TOB1
# 'Th1/Th17'#EOMESlo CCR6lo CCR4- CXCR3+ KLRB1+ GZMKlo/+ CCL5+ TBX21+ 与Th1相似, 差异基因中等表达
# 'CXCR5+ Th'#CCR7+ CD45RA-。高表达CD27,用于形成记忆，不表达CCR7等,表达SELL

In [ ]:
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMB','BTBD9','GATA3',
                      'LIMS1','RORC','CD27','CCR1','CCR2','PRDM1','CD27','CD28','CXCR5','KLRB1',
                      'TCF7','ICOS','CD27','CXCR3','PDCD1','TBX21','IL6R','TIGIT',
                      'CCL5','CTLA4','CD40LG','CCR9','FCRL6','FCRL3'],
              standard_scale='obs',groupby=groupby)

In [ ]:
adata.obs['cluster_dummy']="NOT"
adata.obs.loc[adata.obs[groupby]=="10",'cluster_dummy'] = "YES"
sc.pl.umap(adata, color='cluster_dummy')

In [175]:
adata.obs[groupby].value_counts()

L4_leiden_TOTALVI_0.8
7     49235
2     38120
1     36044
4     34101
0     31649
3     27589
9     23087
5     18025
6     14120
10     7516
8      4684
11      755
Name: count, dtype: int64

In [176]:
cell_dict={'CD27- MAIT':['4'],#
    'CD27+ MAIT':['0','1','2','3','5','6','7','9','10'],#
    'CD56+ MAIT':['11'],#https://www.nature.com/articles/s41590-023-01575-1#Abs1
    'Doublet|Lowquality':['8',#CSF3R CD16
                         ]
}

#存在CD56- MAIT和CD56+ MAIT(8) https://www.nature.com/articles/s41590-023-01575-1#Abs1

In [177]:
check_dict_duplicates(cell_dict)

'无重复'

In [178]:
# Generate new assignments
for i in cell_dict.keys():
    ind = pd.Series(adata.obs[groupby]).isin(cell_dict[i])
    adata.obs.loc[ind,'Celltype_L4_L5_Refine_R3'] = i

In [179]:
(adata.obs['Celltype_L4_L5_Refine_R3'].isna()).value_counts()

Celltype_L4_L5_Refine_R3
False    284925
Name: count, dtype: int64

In [180]:
adata.obs['Celltype_L4_L5_Refine_R3'].value_counts()

Celltype_L4_L5_Refine_R3
CD27+ MAIT            245385
CD27- MAIT             34101
Doublet|Lowquality      4684
CD56+ MAIT               755
Name: count, dtype: int64

In [181]:
adata = adata[adata.obs['Celltype_L4_L5_Refine_R3'] != "Doublet|Lowquality",:]

In [182]:
adata.obs['Celltype_L4_L5_Refine_R3'].value_counts()

Celltype_L4_L5_Refine_R3
CD27+ MAIT    245385
CD27- MAIT     34101
CD56+ MAIT       755
Name: count, dtype: int64

In [183]:
indices = adata.obs.loc[:,['Celltype_L1_L2','Celltype_L1_L2_Refine','Celltype_L2_L3_Refine',
                           'Celltype_L3_L4_Refine','Celltype_L4_L5_Refine',
                           'Celltype_L4_L5_Refine_R2','Celltype_L4_L5_Refine_R3',
                           groupby,'receptor_type','receptor_type_BCR']]
indices['UMAP_1'] = adata.obsm['X_umap'][:, 0].copy()
indices['UMAP_2'] = adata.obsm['X_umap'][:, 1].copy()
indices.rename(columns={groupby: 'leiden_cluster'}, inplace=True) #Save the cluster categorical, check the relationship between clusters and clinical to avoid missing someone.
indices['leiden_cluster'] = celltype + " c" + indices['leiden_cluster'].astype(str)
os.makedirs(f'{finnal_path}/{celltype}', exist_ok=True)
indices.to_csv(f"{finnal_path}/{celltype}/Finnal_indices_{celltype}.csv")
indices.head()

,Celltype_L1_L2,Celltype_L1_L2_Refine,Celltype_L2_L3_Refine,Celltype_L3_L4_Refine,Celltype_L4_L5_Refine,Celltype_L4_L5_Refine_R2,Celltype_L4_L5_Refine_R3,leiden_cluster,receptor_type,receptor_type_BCR,UMAP_1,UMAP_2
D0687_Rep2_CTAAGGTC_TCTTCACA_CCGTGAGA,CD4+ T,Naïve CD4+ T,Helper memory CD4+ T,Th1,MAIT,MAIT,CD27+ MAIT,MAIT c0,NaN,NaN,1.346668,-0.451782
D0365_Rep1_ATAGCGAC_CAACCACA_GAATCTGA,CD4+ T,Naïve CD4+ T,Helper memory CD4+ T,Th1,MAIT,MAIT,CD27+ MAIT,MAIT c2,NaN,NaN,2.793490,1.430817
D0585_Rep1_ATTGGCTC_AGATCGCA_CTGAGCCA,CD4+ T,Naïve CD4+ T,Helper memory CD4+ T,Th1,MAIT,MAIT,CD27+ MAIT,MAIT c0,NaN,NaN,-0.807787,-0.629346
D0462_Rep2_AAGAGATC_CAATGGAA_CAACCACA,CD4+ T,Naïve CD4+ T,Helper memory CD4+ T,Th17,MAIT,MAIT,CD27+ MAIT,MAIT c10,TCR,NaN,-2.901992,0.752785
D0827_Rep1_ACCTCCAA_ACCTCCAA_AGTCACTA,CD4+ T,Naïve CD4+ T,Helper memory CD4+ T,Th1/Th17,MAIT,MAIT,CD27- MAIT,MAIT c4,TCR,NaN,0.635739,-0.702675


# 7. 定 终 Vd1 Refine

In [184]:
celltype="Vd1"
R_data_path = f"{obj_path}{celltype}"
sc.settings.figdir=f"{obj_path}{celltype}"

In [185]:
adata,adt,leiden_data = get_norm_annotation_data(celltype)

In [ ]:
sc.pl.umap(adata, color=['Celltype_L1_L2_Refine','Celltype_L2_L3_Refine','Celltype_L3_L4_Refine','Celltype_L4_L5_Refine','Celltype_L4_L5_Refine_R2','receptor_type'],
           frameon=False,
           legend_fontsize=4, legend_fontoutline=2,
           size=4)

In [186]:
groupby = "L4_leiden_TOTALVI_0.8"

In [290]:
xlsx = pd.ExcelWriter(f"{obj_path}{celltype}/{groupby}_freq_table.xlsx")
for group in ['SampleID', 'DonorID','scDblFinder.class',
              'Immune_All_Low', 'Adult_Human_Blood', 'Adult_Human_Bone_marrow',
              'AIFI_L2', 'AIFI_L3', 'predicted.celltype.l2',
              
              ]:
    freq_table_multi = adata.obs.groupby([groupby, group]).size()
    pd.DataFrame(freq_table_multi).to_excel(xlsx,sheet_name=group)
xlsx.close()

In [ ]:
#数据簇间信息比较
fig, axs = plt.subplots(5, 2, figsize=(18, 12),constrained_layout=True)
plt.subplots_adjust(hspace=1,wspace=1)
sc.pl.violin(adata, keys='nCount_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,0], show=False)
sc.pl.violin(adata, keys='nFeature_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,1], show=False)
sc.pl.violin(adata, keys='log10GenesPerUMI', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,0], show=False)
sc.pl.violin(adata, keys='percent_top50', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,1], show=False)

sc.pl.violin(adata, keys='percent_apop', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,0], show=False)
sc.pl.violin(adata, keys='percent_ribo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,1], show=False)
sc.pl.violin(adata, keys='percent_ieg', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,0], show=False)
sc.pl.violin(adata, keys='percent_oxphos', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,1], show=False)
sc.pl.violin(adata, keys='percent_hemo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,0], show=False)
sc.pl.violin(adata, keys='G2M.Score', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,1], show=False)

fig.tight_layout()
plt.savefig(f'{obj_path}{celltype}/{groupby}_TOTALVI_L3_metadata.png')

In [292]:
sc.tl.dendrogram(adata,groupby=groupby,use_rep='X_TOTALVI')
sc.tl.dendrogram(adt,groupby=groupby,use_rep='X_TOTALVI')

In [ ]:
#cosg差异
cosg.cosg(adata, key_added=f'cosg_{groupby}', groupby=groupby,
          mu=10,n_genes_user=100,remove_lowly_expressed=True,
         )
df_tmp = pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'])
df_tmp.to_csv(f"{obj_path}{celltype}/cosg_{groupby}.csv")
#cosg差异作图
df_tmp=pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'][:8,]).T
df_tmp=df_tmp.reindex(adata.uns['dendrogram_'+groupby]['categories_ordered'])
marker_genes_list={idx: list(row.values) for idx, row in df_tmp.iterrows()}
marker_genes_list = {k: v for k, v in marker_genes_list.items() if not any(isinstance(x, float) for x in v)}
sc.pl.dotplot(adata, marker_genes_list,
             groupby=groupby,
             dendrogram=True,
             swap_axes=False,
             standard_scale='var',
             save=f'cosg_{groupby}',
             cmap='Spectral_r')

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,0],show=False)
sc.pl.umap(adata, color='TRDC',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,1],show=False)
sc.pl.umap(adata, color='GZMK',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,2],show=False)
sc.pl.umap(adata, color='GZMB',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,3],show=False)

sc.pl.umap(adata, color='TRAC',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,0],show=False)
sc.pl.umap(adata, color='TRDV1',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,1],show=False)
sc.pl.umap(adata, color='KLRC2',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,2],show=False)
sc.pl.umap(adata, color='TRDV3',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,3],show=False)
plt.show()

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,0],show=False)
sc.pl.umap(adata, color='CCR7',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,1],show=False)
sc.pl.umap(adata, color='TIGIT',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,2],show=False)
sc.pl.umap(adata, color='KLRF1',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,3],show=False)

sc.pl.umap(adata, color='SOX4',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,0],show=False)
sc.pl.umap(adata, color='TRDV1',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,1],show=False)
sc.pl.umap(adata, color='ZNF683',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,2],show=False)
sc.pl.umap(adata, color='TRDV2',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,3],show=False)
plt.show()

In [ ]:
sc.pl.dotplot(adt, ['CD8','CD27','CD45RA','CD197','CD62L','CD127','CD161','CD16','CD56','CD279'],
              groupby=groupby,dendrogram=True,layer='dsb')

In [ ]:
sc.pl.dotplot(adata, ['TRDV2','TRGV9','TRDV1','KLRF1','TIGIT','FCGR3A','TRDV3',
                      'CD27','SELL','CCR7','IL7R','KLRF1','NCR1','CMC1',
                      'CCL5','TCF7','LEF1','SLC4A10','GZMK','KLRD1','KLRB1','KLRK1','BTN3A1','FCGR3A','TRAC','TRDC'],
              standard_scale='obs',groupby=groupby,dendrogram=True)

In [ ]:
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'LIMS1','RORC','CD27','CCR1','CCR2','PRDM1','CD27','CD28','CXCR5','KLRB1',
                      'TCF7','ICOS','CD27','CXCR3','PDCD1','TBX21','IL6R','TIGIT','LAG3','HAVCR2','BTBD9','FCRL3',
                      'CCL5','CTLA4','CD40LG','CXCR4','IKZF2','ZNF683','B3GAT1','NCAM1','FCGR3A'],
              standard_scale='var',groupby=groupby,dendrogram=True)

In [ ]:
adata.obs['cluster_dummy']="NOT"
adata.obs.loc[adata.obs[groupby]=="6",'cluster_dummy'] = "YES"
sc.pl.umap(adata, color='cluster_dummy')

In [187]:
adata.obs[groupby].value_counts()

L4_leiden_TOTALVI_0.8
5    8492
7    7986
0    6557
4    6181
2    4313
1    3356
3    1867
6     813
Name: count, dtype: int64

In [188]:
cell_dict={'GZMK+ effector Vδ1+ T cells':['3','5',],#CCL5
           'KLRC2+ effector Vδ1+ T cells':['2',],#CCL5
    'Naïve Vδ1+ T cells':['4','7','6'],#
    'SOX4+ Vδ1+ T cells':['0',],#
    'CD279+ SOX4+ Vδ1+ T cells':['1',],#
}

#gdT 在血液中多是CD45RA- CD27+/-

# #γδ T；https://www.nature.com/articles/s41467-018-04076-0；
# https://www.nature.com/articles/s41392-023-01653-8#Sec12 #CD8+ gdT=Vδ1+  CD56+γδT存在
# Vδ2+ T cells were predominantly CD27+CD45RA−: https://journals.aai.org/jimmunol/article/197/12/4584/109106/CD8-T-Cells-A-Novel-T-Cell-Subset-with-a-Potential

# Vδ1+ T  https://www.cell.com/cell-reports/fulltext/S2211-1247(22)00631-3

# γδT细胞根据TCR的γ（包括2/3/4/5/8/9）和δ（包括1/2/3/5）链的表达，γδT细胞主要分为三个亚群：Vδ1T细胞，Vδ2T细胞和Vδ3T细胞。
# Vδ1T细胞主要存在于粘膜上皮细胞中，Vδ2T细胞主要分布在外周血中，Vδ3T细胞主要分布肝和肠。
# 基于CD27和CD45RA的表达差异Vδ2T细胞又分为CD45RA + CD27 +（幼稚），CD45RA-CD27 +（中间，无记忆效应功能），
# CD45RA-CD27-（记忆效应）和CD45RA + CD27-（终末分化）四群。
# 另外，γδT细胞也可分为多个功能亚群：产IFN-γ，产IL-17AγδT细胞和抗原呈递γδT细胞。

In [189]:
check_dict_duplicates(cell_dict)

'无重复'

In [190]:
# Generate new assignments
for i in cell_dict.keys():
    ind = pd.Series(adata.obs[groupby]).isin(cell_dict[i])
    adata.obs.loc[ind,'Celltype_L4_L5_Refine_R3'] = i

In [191]:
(adata.obs['Celltype_L4_L5_Refine_R3'].isna()).value_counts()

Celltype_L4_L5_Refine_R3
False    39565
Name: count, dtype: int64

In [192]:
adata.obs['Celltype_L4_L5_Refine_R3'].value_counts()

Celltype_L4_L5_Refine_R3
Naïve Vδ1+ T cells              14980
GZMK+ effector Vδ1+ T cells     10359
SOX4+ Vδ1+ T cells               6557
KLRC2+ effector Vδ1+ T cells     4313
CD279+ SOX4+ Vδ1+ T cells        3356
Name: count, dtype: int64

In [193]:
indices = adata.obs.loc[:,['Celltype_L1_L2','Celltype_L1_L2_Refine','Celltype_L2_L3_Refine',
                           'Celltype_L3_L4_Refine','Celltype_L4_L5_Refine',
                           'Celltype_L4_L5_Refine_R2','Celltype_L4_L5_Refine_R3',
                           groupby,'receptor_type','receptor_type_BCR']]
indices['UMAP_1'] = adata.obsm['X_umap'][:, 0].copy()
indices['UMAP_2'] = adata.obsm['X_umap'][:, 1].copy()
indices.rename(columns={groupby: 'leiden_cluster'}, inplace=True) #Save the cluster categorical, check the relationship between clusters and clinical to avoid missing someone.
indices['leiden_cluster'] = celltype + " c" + indices['leiden_cluster'].astype(str)
os.makedirs(f'{finnal_path}/{celltype}', exist_ok=True)
indices.to_csv(f"{finnal_path}/{celltype}/Finnal_indices_{celltype}.csv")
indices.head()

,Celltype_L1_L2,Celltype_L1_L2_Refine,Celltype_L2_L3_Refine,Celltype_L3_L4_Refine,Celltype_L4_L5_Refine,Celltype_L4_L5_Refine_R2,Celltype_L4_L5_Refine_R3,leiden_cluster,receptor_type,receptor_type_BCR,UMAP_1,UMAP_2
D0361_M_Rep2_GGAGAACA_AAGACGGA_ATTGAGGA,CD4+ T,Naïve CD4+ T,Treg,Treg CD8+,Vδ1 T,Naïve Vδ1 T,SOX4+ Vδ1+ T cells,Vd1 c0,NaN,NaN,5.494597,6.544697
D0370_Rep2_GCCACATA_CCTCCTGA_GACTAGTA,CD4+ T,Naïve CD4+ T,Treg,Treg CD8+,Vδ1 T,Naïve Vδ1 T,Naïve Vδ1+ T cells,Vd1 c4,NaN,NaN,4.551991,9.690569
D0323_E_Rep2_TGGCTTCA_GTCTGTCA_ATCATTCC,CD4+ T,Naïve CD4+ T,Treg,Treg CD8+,Vδ1 T,Naïve Vδ1 T,SOX4+ Vδ1+ T cells,Vd1 c0,NaN,NaN,6.990400,6.672959
D0633_Rep1_AAACATCG_CCTAATCC_TCCGTCTA,CD4+ T,Naïve CD4+ T,Treg,Treg CD8+,Vδ1 T,Vδ1 SOX4+ T,Naïve Vδ1+ T cells,Vd1 c4,NaN,NaN,5.103128,6.898465
D0150_Rep2_TGGTGGTA_AGAGTCAA_CCGAAGTA,CD4+ T,Naïve CD4+ T,Treg,Treg CD8+,Vδ1 T,Naïve Vδ1 T,Naïve Vδ1+ T cells,Vd1 c4,NaN,NaN,4.557222,9.220356


# 8. 定 终 Vd2 Refine

In [194]:
celltype="Vd2"
R_data_path = f"{obj_path}{celltype}"
sc.settings.figdir=f"{obj_path}{celltype}"

In [195]:
adata,adt,leiden_data = get_norm_annotation_data(celltype)

In [ ]:
sc.pl.umap(adata, color=['Celltype_L4_L5_Refine_R2','receptor_type'],
           frameon=False,
           legend_fontsize=4, legend_fontoutline=2,
           size=4)

In [196]:
groupby = "L4_leiden_TOTALVI_0.3"

In [334]:
xlsx = pd.ExcelWriter(f"{obj_path}{celltype}/{groupby}_freq_table.xlsx")
for group in ['SampleID', 'DonorID','scDblFinder.class',
              'Immune_All_Low', 'Adult_Human_Blood', 'Adult_Human_Bone_marrow',
              'AIFI_L2', 'AIFI_L3', 'predicted.celltype.l2',
              
              ]:
    freq_table_multi = adata.obs.groupby([groupby, group]).size()
    pd.DataFrame(freq_table_multi).to_excel(xlsx,sheet_name=group)
xlsx.close()

In [ ]:
#数据簇间信息比较
fig, axs = plt.subplots(5, 2, figsize=(18, 12),constrained_layout=True)
plt.subplots_adjust(hspace=1,wspace=1)
sc.pl.violin(adata, keys='nCount_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,0], show=False)
sc.pl.violin(adata, keys='nFeature_RNA', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[0,1], show=False)
sc.pl.violin(adata, keys='log10GenesPerUMI', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,0], show=False)
sc.pl.violin(adata, keys='percent_top50', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[1,1], show=False)

sc.pl.violin(adata, keys='percent_apop', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,0], show=False)
sc.pl.violin(adata, keys='percent_ribo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[2,1], show=False)
sc.pl.violin(adata, keys='percent_ieg', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,0], show=False)
sc.pl.violin(adata, keys='percent_oxphos', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[3,1], show=False)
sc.pl.violin(adata, keys='percent_hemo', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,0], show=False)
sc.pl.violin(adata, keys='G2M.Score', groupby=groupby, rotation=90,size=0.1,palette="pastel",linewidth=0,ax=axs[4,1], show=False)

fig.tight_layout()
plt.savefig(f'{obj_path}{celltype}/{groupby}_TOTALVI_L3_metadata.png')

In [336]:
sc.tl.dendrogram(adata,groupby=groupby,use_rep='X_TOTALVI')
sc.tl.dendrogram(adt,groupby=groupby,use_rep='X_TOTALVI')

In [ ]:
#cosg差异
cosg.cosg(adata, key_added=f'cosg_{groupby}', groupby=groupby,
          mu=10,n_genes_user=100,remove_lowly_expressed=True,
         )
df_tmp = pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'])
df_tmp.to_csv(f"{obj_path}{celltype}/cosg_{groupby}.csv")
#cosg差异作图
df_tmp=pd.DataFrame(adata.uns[f'cosg_{groupby}']['names'][:8,]).T
df_tmp=df_tmp.reindex(adata.uns['dendrogram_'+groupby]['categories_ordered'])
marker_genes_list={idx: list(row.values) for idx, row in df_tmp.iterrows()}
marker_genes_list = {k: v for k, v in marker_genes_list.items() if not any(isinstance(x, float) for x in v)}
sc.pl.dotplot(adata, marker_genes_list,
             groupby=groupby,
             dendrogram=True,
             swap_axes=False,
             standard_scale='var',
             save=f'cosg_{groupby}',
             cmap='Spectral_r')

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8))
sc.pl.umap(adata, color=groupby,legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,0],show=False)
sc.pl.umap(adata, color='TRDV1',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,1],show=False)
sc.pl.umap(adata, color='TIGIT',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,2],show=False)
sc.pl.umap(adata, color='KLRF1',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[0,3],show=False)

sc.pl.violin(adata,'TRDV2',groupby=groupby,show=False, ax=axs[1,0],size=0.5)
sc.pl.violin(adata,'TRGV9',groupby=groupby,show=False, ax=axs[1,1],size=0.5)
sc.pl.umap(adata, color='GZMK',legend_fontsize=4, legend_fontoutline=2,size=1,legend_loc='on data',ax=axs[1,2],show=False)
sc.pl.violin(adt,'CD27',groupby=groupby,show=False, ax=axs[1,3],size=0.5,layer='clr')
plt.show()

In [ ]:
sc.pl.dotplot(adt, ['CD27','CD45RA','CD197','CD62L','CD127','CD161','CD16','CD56','HLA-DR'],
              groupby=groupby,dendrogram=True)

In [ ]:
sc.pl.dotplot(adata, ['TRDV2','TRGV9','TRDV1','KLRF1','TIGIT','FCGR3A','KLRK1','TRDV3',
                      'CD27','SELL','CCR7','IL7R','KLRF1','NCR1','CMC1',
                      'CCL5','TCF7','LEF1','SLC4A10','GZMK','KLRD1','KLRB1','KLRK1','BTN3A1','FCGR3A'],
              standard_scale='obs',groupby=groupby)

In [ ]:
adata.obs['cluster_dummy']="NOT"
adata.obs.loc[adata.obs['L4_leiden_TOTALVI_0.3']=="2",'cluster_dummy'] = "YES"
sc.pl.umap(adata, color='cluster_dummy')

In [ ]:
sc.pl.dotplot(adata, ['CCR4','CCR6','CXCR5','CXCR3','KLRB1','CCR10','GZMK','CCL5','GZMA','GZMB','GATA3',
                      'LIMS1','RORC','CD27','CCR1','CCR2','PRDM1','CD27','CD28','CXCR5','KLRB1',
                      'TCF7','ICOS','CD27','CXCR3','PDCD1','TBX21','IL6R','TIGIT','LAG3','HAVCR2','BTBD9','FCRL3',
                      'CCL5','CTLA4','CD40LG'],
              standard_scale='var',groupby=groupby)

In [197]:
adata.obs[groupby].value_counts()

L4_leiden_TOTALVI_0.3
0    520769
1    225132
3    109604
2     39021
4      1439
Name: count, dtype: int64

In [198]:
cell_dict={
    'GZMB+ Vδ2+ T cells':['0','3'],
    'GZMK+ Vδ2+ T cells':['1',],
    'CD62Lhi GZMK+ Vδ2+ T cells':['2'],
    'Doublet|Lowquality':['4',],#像是Vd1,但是纳入过去有问题
}

#gdT 在血液中多是CD45RA- CD27+/-

# #γδ T；https://www.nature.com/articles/s41467-018-04076-0；  https://www.nature.com/articles/s41577-020-0345-y#Sec16
# https://www.nature.com/articles/s41392-023-01653-8#Sec12 #CD8+ gdT=Vδ1+  CD56+γδT存在
# Vδ2+ T cells were predominantly CD27+CD45RA−: https://journals.aai.org/jimmunol/article/197/12/4584/109106/CD8-T-Cells-A-Novel-T-Cell-Subset-with-a-Potential

# Vδ1+ T  https://www.cell.com/cell-reports/fulltext/S2211-1247(22)00631-3

# γδT细胞根据TCR的γ（包括2/3/4/5/8/9）和δ（包括1/2/3/5）链的表达，γδT细胞主要分为三个亚群：Vδ1T细胞，Vδ2T细胞和Vδ3T细胞。
# Vδ1T细胞主要存在于粘膜上皮细胞中，Vδ2T细胞主要分布在外周血中，Vδ3T细胞主要分布肝和肠。
# 基于CD27和CD45RA的表达差异Vδ2T细胞又分为CD45RA + CD27 +（幼稚），CD45RA-CD27 +（中间，无记忆效应功能），
# CD45RA-CD27-（记忆效应）和CD45RA + CD27-（终末分化）四群。
# 另外，γδT细胞也可分为多个功能亚群：产IFN-γ，产IL-17AγδT细胞和抗原呈递γδT细胞。

In [199]:
check_dict_duplicates(cell_dict)

'无重复'

In [200]:
# Generate new assignments
for i in cell_dict.keys():
    ind = pd.Series(adata.obs[groupby]).isin(cell_dict[i])
    adata.obs.loc[ind,'Celltype_L4_L5_Refine_R3'] = i

In [201]:
(adata.obs['Celltype_L4_L5_Refine_R3'].isna()).value_counts()

Celltype_L4_L5_Refine_R3
False    895965
Name: count, dtype: int64

In [202]:
adata.obs['Celltype_L4_L5_Refine_R3'].value_counts()

Celltype_L4_L5_Refine_R3
GZMB+ Vδ2+ T cells            630373
GZMK+ Vδ2+ T cells            225132
CD62Lhi GZMK+ Vδ2+ T cells     39021
Doublet|Lowquality              1439
Name: count, dtype: int64

In [203]:
adata = adata[adata.obs['L4_leiden_TOTALVI_1'] != "11",:]#CD8

In [204]:
adata = adata[adata.obs['Celltype_L4_L5_Refine_R3'] != "Doublet|Lowquality",:]

In [205]:
adata.obs['Celltype_L4_L5_Refine_R3'].value_counts()

Celltype_L4_L5_Refine_R3
GZMB+ Vδ2+ T cells            630298
GZMK+ Vδ2+ T cells            224338
CD62Lhi GZMK+ Vδ2+ T cells     39016
Name: count, dtype: int64

In [206]:
indices = adata.obs.loc[:,['Celltype_L1_L2','Celltype_L1_L2_Refine','Celltype_L2_L3_Refine',
                           'Celltype_L3_L4_Refine','Celltype_L4_L5_Refine',
                           'Celltype_L4_L5_Refine_R2','Celltype_L4_L5_Refine_R3',
                           groupby,'receptor_type','receptor_type_BCR']]
indices['UMAP_1'] = adata.obsm['X_umap'][:, 0].copy()
indices['UMAP_2'] = adata.obsm['X_umap'][:, 1].copy()
indices.rename(columns={groupby: 'leiden_cluster'}, inplace=True) #Save the cluster categorical, check the relationship between clusters and clinical to avoid missing someone.
indices['leiden_cluster'] = celltype + " c" + indices['leiden_cluster'].astype(str)
os.makedirs(f'{finnal_path}/{celltype}', exist_ok=True)
indices.to_csv(f"{finnal_path}/{celltype}/Finnal_indices_{celltype}.csv")
indices.head()

,Celltype_L1_L2,Celltype_L1_L2_Refine,Celltype_L2_L3_Refine,Celltype_L3_L4_Refine,Celltype_L4_L5_Refine,Celltype_L4_L5_Refine_R2,Celltype_L4_L5_Refine_R3,leiden_cluster,receptor_type,receptor_type_BCR,UMAP_1,UMAP_2
D0589_Rep1_AACCGAGA_GCTAACGA_GCCACATA,CD4+ T,Cytotoxic CD4+ T,Cytotoxic CD4+ T,Cytotoxic CD4+ T,Vδ2 T,Vδ2 T,GZMB+ Vδ2+ T cells,Vd2 c0,NaN,NaN,0.221614,2.766788
D0589_Rep1_CCATCCTC_TCTTCACA_GTACGCAA,CD4+ T,Cytotoxic CD4+ T,Cytotoxic CD4+ T,Cytotoxic CD4+ T,Vδ2 T,Vδ2 T,GZMK+ Vδ2+ T cells,Vd2 c1,NaN,NaN,1.483257,5.167009
D0589_Rep1_AAGACGGA_AACCGAGA_CATCAAGT,CD4+ T,Cytotoxic CD4+ T,Cytotoxic CD4+ T,Cytotoxic CD4+ T,Vδ2 T,Vδ2 T,GZMB+ Vδ2+ T cells,Vd2 c0,NaN,NaN,-0.373527,0.131813
D0097_Rep1_CCTCTATC_CCTCCTGA_GACTAGTA,CD4+ T,Cytotoxic CD4+ T,Cytotoxic CD4+ T,Cytotoxic CD4+ T,Vδ2 T,Vδ2 T,GZMB+ Vδ2+ T cells,Vd2 c0,NaN,NaN,-2.335829,5.200785
D0899_Rep1_CAGCGTTA_GAACAGGC_CAGCGTTA,CD4+ T,Cytotoxic CD4+ T,Cytotoxic CD4+ T,Cytotoxic CD4+ T,Vδ2 T,Vδ2 T,GZMB+ Vδ2+ T cells,Vd2 c0,NaN,NaN,-0.532347,2.222835


# 合并所有数据

## PBMC et. al.

In [95]:
try:
    del indices_dict,combined_indices,Finnal_indices_dict,combined_Finnal_indices_dict,Processing_combined_indices
except:
    print("It is None!")

It is None!


## 终版

In [96]:
celltypes=['AtypicalB',
           'Basophil','CEACAM8_Pos_Neutrophil','CEACAM8_Neg_Neutrophil',
           'CytotoxicCD4',
           'CD8Tcm',
           'DC','DnT',
           'HSPC',
           'iNKT','Mast','Memory_B','MAIT',
           'Monocyte','NaiveB','NaiveCD4','NaiveCD8','NK', 
           'Non_NK_ILC','Plasma','Platelet',
           'ProliferativeT','TemCD8',
           'TregCD4','TregCD8','Tfh_Tcm',
           'Vd1','Vd2',
]

In [97]:
Finnal_indices_dict= []
for i in tqdm(celltypes):
    indices = pd.read_csv(f'{finnal_path}{i}/Finnal_indices_{i}.csv',index_col=0)
    Finnal_indices_dict.append(indices)

100%|██████████| 28/28 [00:55<00:00,  2.00s/it]


In [98]:
combined_Finnal_indices_dict = pd.concat(Finnal_indices_dict)

In [99]:
combined_Finnal_indices_dict['Celltype_L4_L5_Refine_R3'].value_counts()
#本轮获得的终版细胞数1742676

Celltype_L4_L5_Refine_R3
Tfh                                  1125265
Terminal effector CD4+ T              717617
Vδ2 GZMB+                             630298
Treg memory T                         239193
Vδ2 GZMK+                             224338
MAIT CD27+                            207265
Treg Naïve T                           76314
Vδ2 GZMK+ HLA-DR+                      39016
MAIT CD27-                             34101
Treg KLRB1+ T                          31848
Naïve Vδ1                              14980
Treg HLA-DR hi T                       14793
Temra CD4+ T                           14511
Terminal effector HLA-DRhi CD4+ T      12670
Vδ1 effector GZMK+                     10359
Vδ1 SOX4+                               6557
Vδ1 effector KLRC2+                     4313
Vδ1 SOX4+ CD279+                        3356
Treg CD8+                               2827
MAIT CD56+                               755
Name: count, dtype: int64

## 过程

In [100]:
%%bash
ls /home/liyanguo/MyImmuCell/05_MyImmuCell_subpopulation/Level2_Refine_R6/ | wc -l
ls /home/liyanguo/MyImmuCell/05_MyImmuCell_subpopulation/Level2_Refine_R6/*/R6_refine* | wc -l
ls /home/liyanguo/MyImmuCell/05_MyImmuCell_subpopulation/Finnal/ | wc -l

8
1
28


In [101]:
celltypes=['Th1_17_2_22',
          ]

In [102]:
indices_dict= []
for i in tqdm(celltypes):
    indices = pd.read_csv(f'{obj_path}{i}/R6_refine_indices_{i}.csv',index_col=0)
    indices_dict.append(indices)

100%|██████████| 1/1 [00:01<00:00,  1.31s/it]


In [103]:
Processing_combined_indices = pd.concat(indices_dict)

In [104]:
combined_indices = pd.concat([Processing_combined_indices,combined_Finnal_indices_dict])

In [105]:
combined_indices['Celltype_L1_L2'].isna().value_counts()

Celltype_L1_L2
False    52030501
Name: count, dtype: int64

In [106]:
combined_indices['Celltype_L1_L2_Refine'].isna().value_counts()

Celltype_L1_L2_Refine
False    52030501
Name: count, dtype: int64

In [107]:
combined_indices['Celltype_L2_L3_Refine'].isna().value_counts()

Celltype_L2_L3_Refine
False    52030501
Name: count, dtype: int64

In [108]:
combined_indices['Celltype_L3_L4_Refine'].isna().value_counts()

Celltype_L3_L4_Refine
True     37417894
False    14612607
Name: count, dtype: int64

In [109]:
combined_indices['Celltype_L4_L5_Refine'].isna().value_counts()

Celltype_L4_L5_Refine
True     42117434
False     9913067
Name: count, dtype: int64

In [110]:
combined_indices['Celltype_L4_L5_Refine_R2'].isna().value_counts()

Celltype_L4_L5_Refine_R2
True     43416820
False     8613681
Name: count, dtype: int64

In [111]:
combined_indices['Celltype_L4_L5_Refine_R3'].isna().value_counts()

Celltype_L4_L5_Refine_R3
True     47317963
False     4712538
Name: count, dtype: int64

In [112]:
combined_indices['Celltype_L4_L5_Refine_R3'].value_counts()

Celltype_L4_L5_Refine_R3
Tfh                                  1125265
Terminal effector CD4+ T              717617
Vδ2 GZMB+                             630298
Th17                                  497199
Th2|Th22                              326494
Th1/Th17                              271661
Treg memory T                         239193
Vδ2 GZMK+                             224338
MAIT CD27+                            207265
Th1                                   206808
Treg Naïve T                           76314
Vδ2 GZMK+ HLA-DR+                      39016
MAIT CD27-                             34101
Treg KLRB1+ T                          31848
Naïve Vδ1                              14980
Treg HLA-DR hi T                       14793
Temra CD4+ T                           14511
Terminal effector HLA-DRhi CD4+ T      12670
Vδ1 effector GZMK+                     10359
Vδ1 SOX4+                               6557
Vδ1 effector KLRC2+                     4313
Vδ1 SOX4+ CD279+              

## save

In [113]:
def save_to_indices(celltype,celltype_to_file):
    L2_refine_path_R7 = '/home/liyanguo/MyImmuCell/05_MyImmuCell_subpopulation/Level2_Refine_R7'
    indices=combined_indices['Celltype_L4_L5_Refine_R3'].isin(celltype)
    indices_celltype=combined_indices[indices]
    print(f"{indices_celltype['Celltype_L4_L5_Refine_R3'].value_counts()}")
    os.makedirs(f'{L2_refine_path_R7}/{celltype_to_file}', exist_ok=True)
    indices_celltype.to_csv(f"{L2_refine_path_R7}/{celltype_to_file}/R7_indices_{celltype_to_file}.csv")

In [114]:
Processing_combined_indices['Celltype_L4_L5_Refine_R3'].value_counts()

Celltype_L4_L5_Refine_R3
Th17        497199
Th2|Th22    326494
Th1/Th17    271661
Th1         206808
Name: count, dtype: int64

In [375]:
save_to_indices(celltype=['Th1'],celltype_to_file='Th1')

Celltype_L4_L5_Refine_R3
Th1    206808
Name: count, dtype: int64


In [376]:
save_to_indices(celltype=['Th17'],celltype_to_file='Th17')

Celltype_L4_L5_Refine_R3
Th17    497199
Name: count, dtype: int64


In [377]:
save_to_indices(celltype=['Th1/Th17'],celltype_to_file='Th1_Th17')

Celltype_L4_L5_Refine_R3
Th1/Th17    271661
Name: count, dtype: int64


In [380]:
save_to_indices(celltype=['Th2|Th22'],celltype_to_file='Th2_Th22')

Celltype_L4_L5_Refine_R3
Th2|Th22    326494
Name: count, dtype: int64
